<!-- dd:dd-lesson-np-1 -->

# Arrays from the ground up

*Numpy · `np-1`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "np-1"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrVXduS2ziS/RWGX7oUKylw4XViex53f6DfygqH7JLtGpelWknlsatj/n1xJ0CCFJEAadvRLakoijiZSACJvOHvN4Tkb/6R"
    "/f3m45G9vbmcnr4d3qyzNx/2l8OFXbn/+83lcH15fvfh9HDgdzx+fT6dr9n1dP7wOdtfsuvb4/fsz+z+Hq8zss7obp3d5+us"
    "WGflbief9PTEf3j3eHk8Xq7744fDnWjm7vtqnV23fx2Ol9OZfdQXt8eHx6/Wn9fT0+Plerda8acdvj8fPlwPD+/Yh7N47F/n"
    "l4No2g9h9eY/62wyEdU627BHIPbrxdE7bYfBFiRvsCB8cdxu42HAN83icHmTYSBZjyBOnX7nYrY4bD8IRsiOkfImr8kyo1gj"
    "9+NlPy3DeNsEPh6HPV5widFC11nO3+cnCIkRLHoprJ287cuEM7Ihfvf2+ENcFSxgV9l0U3shrrMfo+LIXyN6oQWyZVwqt0Ka"
    "K/653iIQoP/ZP10AiLiYGzD9MX2zcdEsiB3botOw4Q0EgCZfCE9BaKzwnE//vgTMBfz2292jlmcxCAlf4STJAQu0A2uMY0GA"
    "qAKUA4A4I4m9N2x6QnGI2CPZ0xr+tJI9VT4wGBgbS/2+Y0+Ow4ZU92HVfUSRT8WT29kLLaRPPu2/vn/YZ+d/ZHfne7Qza+k6"
    "O99zbeT645mRxz6uskf23JfnpwP/iuxWK7NID1Dub32d8XmaruSYB88728IvvTMTxGdY3Tb/nPPZVpJEDEnhk9mwxjwrOU6z"
    "jAo+t8R0zIgGPSsdbruMEMp1nAhC0HbRfhDNSf7jrhSp+QAnmQ9yMadVIaRNwK+fGqpYosQ4UDCCDeXLYFoU8pmhSJg6I2YW"
    "NjXyl5xJRFpYngYkRiVgNImA6Rl/utL+h3jG9vF4LfM/+MIQxDVNUniDH59O+yslwU2iTs8ENhYon7mYnblOhOE8tTZFRRq1"
    "oqM+rns7ogCwVC2fcgWtwybqdg8S2CIWcxZXQ2hoi0JtE8RLyuWeWGibEMLVWhW4Q+ZyyH7IFw4Q5ZzXpJWLco7NMlhge6MG"
    "oiRKVQ3tYkZr7wJs297VJpZHgeboDSE7hFSxsnNkEHNntbtcz2pBO662D1y3Mua/4017Xw8vGyvclGS/TN4OcmxoTmxBSPDM"
    "XAoC0yzbZYMvQaDpYqDNZp5gksKatM4+nJ64aYKvGQNUtPd1ybG/mUbX+9PpiRN1325H5Cuf4HvXguwrhhS+Fi1NChBq/jOg"
    "Kk733sHcLn6m4CQQGL63+jkkuFLf+as13CXZp1+3V+Hz4qaNdbbBls2Q/dlwtVNQ+OeVL9KUrAZY8r3LCctZts6+T2SGaOKP"
    "deuW0R404Z4RzksvyBANpaVY7FM3XGWalyrPtC3gW81D4DvmXiEb85KhJdV2Y6i3VkZdOCFk7c/746fDHckdiSvz1fZ8uHze"
    "Px/UDiafXQSFKnrvuAZbMez/tVtn0+/d6Sf7Df7sr1puwpHYiIvbMfej8r0bXxjE7g+zmzHb/mLucRDeB8K3IPyZfHPL96gw"
    "meKOPXvMC9Et8wXGh2I62iqGCiCt5l+n0Pw7uoxWvh4v7wSEx+Ond88nJgB3AaqZFHu1CbSHt/iMuxf01V2MMp4aeBgYsgAX"
    "bQaGoSuW6uOR3vZ81f9+19964OGvYMzIF2WGnw9+Fvipt92SsQP+mzOxUeGdJcJdy61Zu9Xb48fHpyd208Y3ur6tM/61PZ19"
    "u8UZrZwQ+cZXd6xMVJ22J3ahS4KNGaWCLABCEeVcEWtBNalA5cqlLp8fBE1pEZXVvUUqWExGN4XsXu7WkG+lWuHVei6ui8sw"
    "lmpFocVPEsInHZW6XV+LX96yhs30oV9+HXsW/n3NfGTZPpv6EkRE9VNpMKOoiI6kObJxLA1AZOub4vn348u3uGHiCk6GV3Ay"
    "vIKT4RWcBOknkla+UC5AawGAJpbzbTE7to20BAgWbhRn1V/mG+ua+SaYICL6aXZ6fHITjFVED5D5mc8bGVXqPTe4ou/e4BkW"
    "5oYhvZ7fYG12MU0wjzw+8J09NyPkfh6yG25xUdwyjY8D25UEnhRDyjpDS1CCIeCK5cCl4Wbxc+UiHDBd+60gswDGPccapWmt"
    "7dgspSa4c+UhL8yedp+bBxLNaC003bYg1sEiDcpCwiqCcajNXblFaVAIXnQY5rFkOKwT6gqQfxtLrqLxazVAqgY4dCZwkRET"
    "sofToNPhyyaGuduCHFdVdDTQqyMZrrPAoUMEyt49G+TZx9M5e84ej2qaeB0OmiVie49UQlIugsCUMypXwfXKAjBdzXFwu04O"
    "kgq3Bi3CRAVi7XLgcEOxGh+BCYRLhZOKB2pvR/t0GDo7aC4ZJ7uReNRcwMA+x6Tt9J5bK56Zqusr4VgqtMcUS89SIyInOR2O"
    "t6nni7L8VNYut0y8FNqWM274+tJ11ZjMqi/tJOQlvw0KhkyCgiuqfQxrv4G1rGH/ROKN2VJCQDAIO6Mv4bn1JbEQy4REhvly"
    "3Z+vTOO7np7Z7m//9HKQW4a8t+U1xPR/cpPBHmXWgIBoMuzHQ9iRciAkAy9VBP8rCHw+DF4GWVfJsPd1MeAIHxEWOddR194T"
    "gZny6H2gdjsAkYqZmczFVudFj+SqWExDu5Ornl7htuI2s417vbmD48soDza/NxmrOlIIARSKCapZNGCiAeNWLyOJ9bIUXMWr"
    "SJ2nq50lhogtPpI59LFohLnMABHqWaWVLSLGM0ZtMFujhCAXClqrclWJY/28lVfeHj+cngZ1H/bdkM6phhsT5xUw7GgUDwrF"
    "o4Z9BB6ZWgTnSC1zLFcw+4YjiASOAgu1upBpqiuz586TzuiYiOXq8Gw5TTz2g1e+MombbEPC601DwgbrPcgGa+c6/1SJoIoN"
    "tjcvHje8vrHd5IAmh8KhccAUDSWxjQ9s3fMtHTDAlQOYPapJCLgR+BqBr3HmLXAUhLtM5LueTJGUMkVk/jZsnUAONm7QQiml"
    "gT3N5aO40AryLUE3qwaum1lVN89G2V0mxX6xs1I6e8gl9Tg/ejQDeqKXem0TVFmKQXXCJmAnc2LXdk6ppQhrUhr9WWIv5sEe"
    "r0wPKYMew0wi2EYzrKW1QlWmEYGCXPTl4CfafFc5xjhCose42jYejg9ypq0Hgmna+7pRNfY3gUH23flrcm85qHnuOlkSNm9w"
    "Q2VJE66C6JEOQl8M50rNAr4AMpmXcKGLcpm3WMowS8lxGHI0nE83C26kYnzlCM1J8tSF+/Pp5fjAo0BL6Rn41noGrBC33ZDV"
    "R4ajiJdqspGqF0QZiyKo5Twp/ZxyYRku+Uu9g6YixwLhU4dEQw2kwuCqBDj+0uyg+RmxCKn4JyDxf5Ux8OM8fuHhhkquI4tA"
    "JWlRL6aCb389gQqkGKtCNgtj5iyUP0EVqhGehUIZ9Qv+qZTm/UJYs/lLIeoe8he+7eGfGvFkNN0m7RIuwZEZCXeGu/S/Q5BS"
    "ZS2eCyjIvu8gVOOGzMjKduYAIdTW9bkQWob7Ir5Q9OfHT5/X2dPp32qf27c/mTtuuRrUfkI5+ieyzm6frn1u0cntU0ibqGt3"
    "CGwUSe1wQyCNc8W/imhc+/lFETYlEjSFKuJxkB0neZpiZqLewhbScGA7FE5gOFEYw1vDSP7DeYMwLrGYnYi6SJoaEUqImK2o"
    "uhE3BDWkbsQ8mas7i6ZEeZm3fVOqJ9A6x3VRNeJaI/7VNaoaVAnFpXYeKu6q5V1ViYsa5zwhOZAdZYRw+Qj30egjr0cLbmdS"
    "QtEsES/AfH1+33d1Q1gWi1lcTaUluWeBOUnQkvhRKtR8I1nJHTtZkoBNLvVHEwRNYinSrokkFTqgneILkXXIYncCq3MgsXyi"
    "RcWMNbdVzXq7R+19igQJGgLZO4ZSKldv3whmvX0zlMtm/aKX1uZ8F1hqBeZs8hGg6jgsRoJVN2JIEoHkYNUfCxLTWo+goHPT"
    "BwuK0ehsAKWEKvbjckn249IaDm3sBU7jRHs9nE+XuzvhhEarVV/NefUD/INpMNn7H9fD5Y8gR4bb3nhFl3EANQyA9l2NLU/j"
    "DZMc0q6nPNIupNEIZusCRd0KRtNbZ+qnaV+IX12lET+jtEmbo9T1C2lbXL09vn88XgaKp7yuM/6tGmT9RCvL2yXvuzHg9PkE"
    "0m0ia8JjYJyA0vwbmwayLA1w6KSwUOcLopZTnC4AAcLeNVxblOAFKcnhciNkH4vXWo4DuzswWliK7BAg/R+2/Ui4STsR4Oy/"
    "MvIvNmGx9/xfqRJlcLvDlOUFQpSwTh8xYCgZsDapDwiHJ9xzRNuCMY1km4yyd44RJ8O4aT0TBi53fdLQpFkXuYAtwG5Yj6cC"
    "W3RTJTcEjBALNvJtF4OI0wrjxk6C3di5h3mZ3DE70WDliwySJwipQ76AHtnJze/mNrtKH7gMHqJBrRUR/FR7WHVunBMxJ4M/"
    "udlF11IUV7m99kYtRfZ/vgO7qsMpuFERkqN260FqMyWNlue9GwIlE1F4SBO7rg5OGMnY2LP1T8Q/dcKg2utTIrh0NJS7nbRP"
    "QZIZ2CJ7sTQ2tVreIxyy0peMRT25yTOSQ3vtI53MTzpxSTerKNUEG/L5dKGIhxJZukTi+fsXh/QvjCjcI0qcdDQvUVhnyMjy"
    "pTDkJO8J3XgaShKhU/vVbn8YJc5JtJTixmckGffQGWuyVq34W6os4sdYiSsWD8BitGLxANwoEZeua9m29JBTzUe5UjdoifyX"
    "wVOQ7COQxImwUw5B6kbCBx7IY4FVWU264sBuPqhwkFViTFUEv4YrAKRilXOeVZMo1dkEe4t8n8tgHrhbXX66hgnLRm3RkDg0"
    "RhtUYw6GJ2/x5NHcCdFNPRNIe/xQCwrDQdliq4411P7nxVJwe/mMg4EtdK3S60WORurM2uk4ZHYC3UWlzKrzZMMBWL9PXa0k"
    "DMEuecrr9C7I5cm3MhFMwCnldsXkupqpkiRwnKpjc8u1d2cqv59zfmwhYOINSgvEMOEUgGBk+dq7DZ0KTOZ4W5IZDIB4wwVD"
    "OLNrpYYurf2NH1rm0+3kVBql4JnVBIhFHesq5lIYkCao5SYmx9ma/PPuBu1Gu86wkb9udbF8BlGx1vkQ9jSORcr+Eygm5lTw"
    "UBh6UjGf4xTrcDYYBuxihWVAUxht3hru9p+tZ4XMVatoQ7vV9HqBSiOVhmppVq+g1W4QsHEEb1BWKy6AzWIif8/ewRWJYPQq"
    "y4fgOJzhG6suOJT5zjMQaqNQ89QOQO6/pTzJiHGcwnRPE2sVUkTqdYqQviYU0o4jT2T9bIzlEdL4hrb1swgUSLNtROgxwiAM"
    "jXbAwbigY0Dkyb4gJmCnoq8S0zJxwMqGmMh6OlCa9DWw3IbzQB6a220iiRyDwUX7qZUz1rf8RHGsdfh6GgH5fb1jEIbR9iRY"
    "zwWODYREhAvqjs04/iGdsJFXeVPlBaa6rH//m6EftAckVTNoLRtl+4et4FIedKA71LLmcQntQHDaCBTtLau3aPeb6FJ6giZF"
    "hBohk6yVvNRp5cWKVtnQHkxo1ejCuJ9IoWaabisgqUJaNHcpcMrH5drp5TwcVli9SFNYvZCF1aFs2sgUvY2oC5CkRjnRVQbk"
    "RGc/XcklTRO14ExjhV3WXYT1ueWl3ESByj+29+vs/a1hlrutBcx6+/700oWJ4LgQFIha2Qsfns3onnYKLPV0GorOKZfr6UtP"
    "Bf+YDqX9RBIQM039+x5qVXmggMI0vzeze5N4dqdGJ1ZFc59O6+zzo8iyR70zm4ynS941PVu2iNcT+uAwikJnPTrCDLAhHb7J"
    "pSwOminZXACrfHcwqZIXMZBya+9H0+79qCyBQvSZG+F7VGrX6YYaKJplDRTdFCqgXSa0kkYHhFI/kah8QmAWAtxuHCuwlYSb"
    "SWA90DRWDnfiQwvuqTq9HSn+6DBoUQ1mU3MVJ8ROLn4hfr0CT4dBDWozI7A5a47jmh0dUK9vEJ1rM1oYijbdiqwCmtPSCD50"
    "6FbcxGjDhWhYCmO97GFSssCMxsjrzHhxMqUizbFQE9vjOgRofSVeu9HEVlFgq6bsa8e5MLG5HMjUVlXW6jigcf5zvTbnM+yz"
    "vLurMcu60G3Hzl/jryuQsn0bkKzwsTwsn/ofyh2VKybewoDIyZIaDPrvX6tzijh+KA9Zkdr7f9sW6YSUrrPHyyMvPnn8cOB/"
    "Xbd/iUetboeWSl0pqH97R7jNg0zsNSOAtS7NmVgXy7l7bLLt5upcm4VKVsvlZXVcSxH9nK/g+8rpTcnEgBhTrrX7D9R5KVSK"
    "iUnICWpUJjfofi9J2sMXOidh8sRk46NaMYHN7Lzk8aNP8wKYX7pLhwEIgddipglhoBWg/EM+VvZhSThOBSXCX2ivilIMnBKh"
    "dhqbwwtZqPQiri1cP/NN1+npQVqrvIYqc88obAKsQ2WpLjaYAghGaS7QjbcvDWvXxYaiGAVcA2ifRzSeR0rM6rnMN+KQTHNA"
    "e5D1whh/eCeQItGZyuONFmZ8IKjtRtqh8ZCPf4BY/SNhYwCv3MTU/uwrW7eWUee3pjxTnbo+jigbiVVlnBSxH94qnb66lN3i"
    "mVIn9d3pfWTv5+CgnETxOCoUB4jDPcY8YYSQifvqtgAPtypaW02iODA7whBZSd8D7UGgk/kYTKzcdZvBqjB1dDbvcf/1cOno"
    "OlzV4ZqOODrqw+ncuUENbBFxiUd2eTL5xZTkzq2S3Gdx7qHe7AkMTGMSTQ3yhCCVCqV2mI0pID59fHqoFaH0A3R6w4bmpFHn"
    "SNhNQymrBsjqi+iMBFWhM5e/h9SJzt5OQqYK25I9JRNQsNNZZgpRgzPxwc2dgSfCk/T+nYIj+1TxOH3Kk0oXiwwjgAKBNt2W"
    "WAA2LueUWMoH1kNgyGV79LxOsTTV53GZWlcr0i5gdgRRYQZJpIbwq+hTdsw2ShgX3vEE91oB6340Yc+6D9SyTnuqCa4W205M"
    "z+8WM11qHXpi83o8wyyD0NyenxU4o6csIw7pzQ6dokeNU8EKuAZ1J/LK5LI14CC2+8iFIDKG7h7fsrZPAGAeEBMzju0koQIM"
    "x2zlOg8zotbMMvM0ZvoRmvWX4fPB8t7hQ+75lYPHDjXy6SmGrOeUzWiA7hwIncfamOgvw+fgARGCQxM7++s5upf0TpUiM02J"
    "TuKWU/gOehD8vf95evXH2xjNfSpmHIr59rOB1QPa+QvMz8KatsBrSmPRVsfxClulH4m13sl6mMVMp/Z4ZikD9ssEfc6caE90"
    "SQgaUMHne99e8mWQdzfhVOAqXF8GS4J9maBmIZcNcAbIMuZRXdKtvA2N+udFhBSSHIZEnBZn/V+Y/4xQ5zOGju7iurUThgqK"
    "Qvc4x6L6tvWXBWeaWEUlooYYSsQM1Fno81g4Y59iMl+sNLEoeeo8zSq4YHLIZDh1M/OY+NFNjRDOhu4eU9ZmVSVaP5mirJ9a"
    "44TQwD61Ghhjwo/B4qx3plRub3/nXtixm+y7kVWXFTkVVlvguxU4GKLPELc87Kw8obrSrmUOU2+SDeaGtkiy2hGDabbH3Y9O"
    "BY+ZycW6JDRqN+OcSnO9aSNAVjEJYS5d9QJ0yV7KbZLkpRpAjZqqqZiqf4yVBEpNSa6HXa+cdegFyQb3eXbNDasuUNe7Sro2"
    "Tap5qM6ri7Zqvn86ffjSsY4oR5L0gDC+nw/Pl3fndSbeP8gagGSwerV4orrZ/Khbynrgplt1rXNd1NqcdGOQdt0Tnu8m26w9"
    "TKn8nKCdaoiLccJMh8KlqV5iCDRlcwc6/OeQ2S5+bam3FmkEtbrgnYdY3Kn9+VOIxW1hZSJ8beHEDtTY81P807qXmPHcnipi"
    "Yqqb2Tb6DOFhf+259qksZejTatUvppXtjd385mrL6YOJojEWAZWde4VE/aBoHKJGVMVtwDaLge7E5qytCGxWWVKYJUcdVjMA"
    "sz0oLwJjJQoyN3rjhItoE9m3/dPLYWB8cLPey7FPB+4F60s65LO4wY//apohjZpxNJXrHsTFANY8HqVjwYHiw8PsREl4Gc/C"
    "RkhvNYByZICFAdXNwIHKQTQkmiQeonV2nz7DM3aQnY+fBND/PRwP5z375m61/bo/vuyf3l0Oh4e7nC/Xx2HPWdnz/bAnrrPj"
    "BOdeXRNSCg9fgwuEhE+R1qQuJxe8vwUerTwnmSXBnjclKapUOKuV54CyJDgLmjdEFPzDTV0j8aksGiLDusuirEUHEEqqkqQi"
    "BxO68pzvloQg0pRYgi8wg58MM0EkX3kOWUvUC7jiZ8ExmaeYlLWQ+QrXVHzCuKGoPcgg2glqjnlS3p/BEFCpY5+NIn3unzvK"
    "Q6Den05Pd3fn7J9/Zmi1Zc+5W2X740PGLv032wLKK/q+8/b48vXAb/mTjbuMMezM+Pyd/f1P/unxyG41kaWy4ZENJpX7kTZF"
    "PCxRfPysrt+HE+05YNGckDpS/rtyQsNPIxPF7y0TRTqZ6J9B+FvJhNrFezghNyMoT3Ie6xSNI5KJ7IaXr3cPj1//RCvnFDF9"
    "GduXmW7JeHHH0LMnf3zaX5m+eWcZPVoW8uNRjp9G51rDwL49uHcBhQYuCzVsgurwm/LPmZ49HqQQNuVTFMbfk025PXe7UuWX"
    "tTiBI5P2N78nK50p797HQDDbittsa5rflG3cXDImgSNi6GWqOkWvSVsBa6jiSrfCkfGP26HZ3UIznYNd33sOdX0fYCjHjT5M"
    "VWQYCXQFCilQLWk9HQ8XMXHSlaFLXqPy3KX0NBhHMA2HO1irT2O3HMX2yXd4ps5wQhFwHk7Q4QcjoC9RjRX3WGnpmleihpqE"
    "VAO7J/6BQueiwsiUBq00zirxjEAsfz0anhHMWQSrqbHaLpk3It13wSJGB2vmeXJponGCYuFgc29hTqeehwQgej/Y9BIBR0i6"
    "qyz1QqY6sjItbrTlB4puc5N1gOpfd+2eXrVdiaMO6Jd8DUlQshZl5aS3azoOnI03CRnqzlwoGNat2X2o4sskfBgOB5s0Lg+o"
    "fKxS3iRkeVv62Ehr81tI69DhJoNSmlKzHNBsJlcxHKUAe9dhO5a2+/zwiqrOAGyHZCBS/7izAwDdb1YR07n9sIEx6t2iBrPf"
    "3xonylcdZmelX6OZBo4JnW4PbCmHKBuXqtwCX1hyVEoS+W5i+vG7+2k13Ech7dS5vzii1eEKGONN6/xaef5xKAI5BQT3AOps"
    "8FXrZNWKEV5q/vUkDuDB2s7fR6sTyJp+jJKofuwxZ+XJqZHSGgjSWlsiQLpBywMsHMvhH4Oo1qyQQ6mdGtlyRv/ejfFXtSjD"
    "sHTjt9nTLQElv9LG8/aMTlv5D68B7l/psSrAGFr42/phTEFyvwF2YIODwzlm7Z8iQPp7Mbx6/D2y2aVkMPqwnq/dGMXhCXpI"
    "J+hPAP4TLCSFvMGRIdf+1jnv0VoeJvWBIUsodVSGda6Gy8KCcKqVRL8pzQGHg2yVUR/Kwrs/n4ZySDUFclNOA9uX4+X/Xg6H"
    "14MI/OqEwI5MT7fQVnZ/Y6MO/Of/Ac9mhs0="
)
print("Delta Drills checker ready — 84 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-numpy-ndarray-model -->

## What a tensor is — data + shape + dtype

`numpy.ndarray-model`


### a tensor is one block of one type


PyTorch's core object is the **tensor** (n-dimensional array). Everything else in
this course — indexing, broadcasting, einsum, einops — is a way of manipulating
this one object, so it pays to know exactly what it is.

A Python list is a bag of pointers: each element can be a different type, live
anywhere in memory, and even be another list of a different length. A tensor is
the opposite: **one block of memory holding elements that are all the same
type**, plus a small amount of metadata describing how to interpret that block.

By convention PyTorch is imported once per file as `import torch as t`. That
short alias is what the ARENA exercises use, so every `t.` below is the same
library you would import as `torch`.


In [ ]:
import torch as t

# A list can hold three different types at once.
print([type(item).__name__ for item in [1, "two", [3]]])

# A tensor holds one, for every element, and says which one.
a = t.tensor([[1, 2, 3], [4, 5, 6]])
print(a)
print("dtype of the whole block:", a.dtype)




A freshly built tensor lays those elements out contiguously. Operations that
only re-describe the block — transposing, slicing with a step — hand back a
tensor that *shares the same memory* in a different reading order. That is
cheap, and it is why `.contiguous()` exists: it is how you ask for the copy.


In [ ]:
at = a.T  # `a` is still defined — this cell continues the one above.

# Same numbers, same memory, read down the columns instead of along the rows.
print("shares storage with a:", at.data_ptr() == a.data_ptr())
print("still in reading order:", at.is_contiguous())

packed = at.contiguous()
print("the copy owns its memory:", packed.data_ptr() != a.data_ptr())
assert t.equal(packed, at)




Why this design? Because when every element is the same type and sits at a
predictable memory address, PyTorch can hand whole-tensor operations to fast
compiled kernels instead of interpreting Python code element by element. That is
the entire performance story — and the reason the idiomatic style you will learn
here avoids writing Python `for` loops over elements.

The general procedure for turning existing Python data into a tensor is
`t.tensor(data)`: it walks the (possibly nested) sequence, finds a common
element type, and copies the values into one block. Nesting of any depth goes
through the same procedure.


In [ ]:
cube = t.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print("shape:", cube.shape)

# A cell ending in a bare expression prints its value, the way a notebook does.
cube * 10




Two ways of reading a tensor back out come up constantly from here on, so name
them now. **`t.equal(x, y)`** answers "same shape AND same values?" as one
bool — the whole-tensor comparison, which is what a check wants. **`x.item()`**
pulls a single element out as a plain Python number, and it refuses unless the
tensor holds exactly one; that refusal is the point, because a silent "first
element" would be a guess.


In [ ]:
same = t.tensor([[1, 2], [3, 4]])
also = t.tensor([[1, 2], [3, 4]])
print("t.equal ->", t.equal(same, also), "  one answer for the whole tensor")

one = t.tensor([7])
print("one.item() ->", one.item(), "as a", type(one.item()).__name__)
try:
    same.item()
except RuntimeError as exc:
    print("four elements ->", type(exc).__name__, "- item() wants exactly one")


Turn a nested Python list into a 2-D tensor:


In [ ]:
import torch as t

# A nested Python list: two inner lists of three integers each.
rows = [[1, 2, 3], [4, 5, 6]]

# t.tensor walks the nesting and copies the values into one block.
a = t.tensor(rows)

# The values survive the trip unchanged — .tolist() reads them back out.
assert a.tolist() == [[1, 2, 3], [4, 5, 6]]
print(a)
print("back to a list:", a.tolist())




You never told `t.tensor` how big the result should be, and you never told it
what type to use. It read both off the data. The next two segments are about
those two answers, because they are where almost every tensor bug lives.


<!-- dd:dd-q224 -->

### Problem 224 · faded — your turn

Turn a nested Python list of equal-length integer rows into a 2-D tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 2, 3],
        [4, 5, 6]])
```


In [ ]:
import torch as t

def solve(rows):
    """Return a 2-D tensor whose i-th row holds rows[i]."""
    return t._____(rows)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(224)


In [ ]:
#@title 💡 Solution — Problem 224
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    return t.tensor(rows)


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


### nesting becomes axes — shape, ndim, numel


The **shape** is a tuple giving the length along each dimension (axis). A 3×4
matrix has `shape == (3, 4)`: axis 0 has length 3 (rows), axis 1 has length 4
(columns). Axis 0 is always the outermost nesting level, so a flat list becomes
1-D, a list of equal-length lists becomes 2-D, and a list of those becomes 3-D.


In [ ]:
import torch as t

grid = t.tensor([[1, 2, 3], [4, 5, 6]])
print("shape:", grid.shape)
print("axis 0 (rows)   :", grid.shape[0])
print("axis 1 (columns):", grid.shape[1])




Two smaller readings come off the same metadata:

- **`ndim`** — how many axes there are. This is the nesting depth, and it is an
  attribute, not a call.
- **`numel()`** — the total element count, which is the *product* of the shape.
  This is a method, so it needs the parentheses. For a 2×3 tensor `ndim` is 2
  and `numel()` is 6 — six numbers arranged as two rows, not two of anything.

Getting those two confused is the classic first-week error, and it is worth
fixing now: `ndim` counts axes, `numel()` counts numbers.


In [ ]:
# Four numbers, three nestings, three different answers for ndim — and the same
# answer for numel every time.
for data in ([1, 2, 3, 4], [[1, 2], [3, 4]], [[[1], [2]], [[3], [4]]]):
    x = t.tensor(data)
    print(tuple(x.shape), "ndim", x.ndim, "numel", x.numel())


In [ ]:
import torch as t

flat = t.tensor([4, 1, 7])
grid = t.tensor([[1, 2, 3], [4, 5, 6]])

# One nesting level -> one axis. Three numbers in it.
assert flat.shape == (3,)
assert flat.ndim == 1
assert flat.numel() == 3

# Two nesting levels -> two axes, (rows, columns). numel is 2 * 3, not 2.
assert grid.shape == (2, 3)
assert grid.ndim == 2
assert grid.numel() == 6

# shape reports as torch.Size, which IS a tuple subclass — so it compares
# equal to a plain tuple, and tuple() converts it when you need the real thing.
assert tuple(grid.shape) == (2, 3)
print("flat: shape", flat.shape, "ndim", flat.ndim, "numel", flat.numel())
print("grid: shape", grid.shape, "ndim", grid.ndim, "numel", grid.numel())


<!-- dd:dd-q482 -->

### Problem 482 · faded — your turn

Count the axes and the elements. Remember which one takes parentheses.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2, 6)
```


In [ ]:
import torch as t

def solve(rows):
    """Return (number of axes, total element count) for the tensor from rows."""
    a = t.tensor(rows)
    return (a._____, a._____())


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(482)


In [ ]:
#@title 💡 Solution — Problem 482
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (number of axes, total element count) for the tensor from rows."""
    a = t.tensor(rows)
    return (a.ndim, a.numel())


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


### dtype is a property of the whole block


The **dtype** is the single element type shared by every entry — `torch.int64`,
`torch.float32`, `torch.bool`, and so on. There is exactly one per tensor,
because there is exactly one block of memory and every slot in it is the same
size.

That has a consequence people meet by accident: when you build a tensor from
mixed Python numbers, PyTorch cannot keep some entries as ints and some as
floats. It picks ONE type that can hold everything, so a single float anywhere
in the input turns the whole tensor into `torch.float32`. All-integer input
gives you `torch.int64` instead.


In [ ]:
import torch as t

print(t.tensor([1, 2, 3]).dtype)
print(t.tensor([1, 2.5, 3]).dtype)   # one float decides the whole block

# Say what you want up front rather than relying on how the input is spelled.
print(t.tensor([1, 2, 3], dtype=t.float32).dtype)




Ordinary division still works on an integer tensor — `a / 2` quietly hands back
a *new* float tensor — but anything that has to write a float back into the
integer block does not. `a /= 2` raises rather than silently rounding, and the
error is the useful kind: it happens where the type is wrong, not three steps
later where the numbers are.


In [ ]:
ints = t.tensor([2, 4, 6])
print("ints / 2   ->", (ints / 2).dtype, "(a new tensor)")

try:
    ints /= 2
except RuntimeError as exc:
    print("ints /= 2  ->", type(exc).__name__)




Two tensors can hold the same numbers in the same layout and still disagree on
dtype. Shape and dtype are independent, and code that checks only one of them
is checking half the question.


In [ ]:
import torch as t

ints = t.tensor([[1, 2], [3, 4]])
mixed = t.tensor([[1, 2.5], [3, 4]])

# All-integer input -> one integer type for all four entries.
assert ints.dtype == t.int64

# ONE float in the input decides the type of the whole block.
assert mixed.dtype == t.float32

# Same shape, different dtype: the two questions are independent.
assert ints.shape == mixed.shape
assert ints.dtype != mixed.dtype

# dtype= overrides the inference rather than relying on the input's spelling.
assert t.tensor([[1, 2], [3, 4]], dtype=t.float32).dtype == t.float32
print("all ints ->", ints.dtype)
print("one float ->", mixed.dtype, "  (the whole block, not just that entry)")
print(mixed)


<!-- dd:dd-q484 -->

### Problem 484 · faded — your turn

Two tensors, two independent questions. Compare each piece of metadata on its
own.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, True)
```


In [ ]:
import torch as t

def solve(rows_a, rows_b):
    """Return (do the shapes match?, do the dtypes match?)."""
    a = t.tensor(rows_a)
    b = t.tensor(rows_b)
    return (a._____ == b._____, a._____ == b._____)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(484)


In [ ]:
#@title 💡 Solution — Problem 484
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows_a, rows_b):
    """Return (do the shapes match?, do the dtypes match?)."""
    a = t.tensor(rows_a)
    b = t.tensor(rows_b)
    return (a.shape == b.shape, a.dtype == b.dtype)


example = ([[1, 2], [3, 4]], [[5, 6], [7, 8]])
print(solve(*example))


<!-- dd:dd-q523 -->

### Problem 523 · guided

Write a function solve(rows) that takes a nested Python list of at least two equal-length inner lists of at least two integers each, builds the tensor a from it, and returns a tuple (view_shares, copy_shares, values). Take the transpose a.T and then make a contiguous copy of that transpose. `view_shares` is True when the transpose reads the same memory block as a, `copy_shares` is True when the contiguous copy does, and `values` is the transposed data as a nested Python list. Transposing only re-describes the existing block; .contiguous() is how you ask for the copy. Compare buffers with .data_ptr().

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, False, [[1, 4], [2, 5], [3, 6]])
```


<details>
<summary>Hints</summary>

1. Two of the three answers are memory questions, not value questions: does
   the tensor you got back read the ORIGINAL block, or a fresh one?
   Transposing never moves data; `.contiguous()` exists precisely to ask for
   the move.
2. `a.T` is the transpose and `a.T.contiguous()` the packed copy;
   `.data_ptr()` reports which block each one reads.
3. `view = a.T`, `packed = view.contiguous()`, then return
   `(view.data_ptr() == a.data_ptr(), packed.data_ptr() == a.data_ptr(),
   packed.tolist())`.

</details>


In [ ]:
import torch
import torch as t

def solve(rows):
    """Return (does a.T share a's buffer?, does its copy?, the transposed values)."""
    a = t.tensor(rows)
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(523)


In [ ]:
#@title 💡 Solution — Problem 523
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (does a.T share a's buffer?, does its copy?, the transposed values)."""
    a = t.tensor(rows)
    view = a.T
    packed = view.contiguous()
    return (view.data_ptr() == a.data_ptr(),
            packed.data_ptr() == a.data_ptr(),
            packed.tolist())


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


<!-- dd:dd-q480 -->

### Problem 480 · independent

Write a function solve(rows) that takes a nested Python list of equal-length inner lists of numbers and returns a tuple (a, shape, is_float). `a` is the 2-D PyTorch tensor built from rows. `shape` is a's shape as a plain Python tuple of ints, not a torch.Size. `is_float` is True when a's dtype is torch.float32 and False otherwise — remember that a tensor holds ONE type for every element, so a single float among the numbers changes the dtype of the whole tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([[1, 2, 3],
        [4, 5, 6]]), (2, 3), False)
```


In [ ]:
import torch
import torch as t

def solve(rows):
    """Return (tensor, shape-as-plain-tuple, True if dtype is float32)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(480)


In [ ]:
#@title 💡 Solution — Problem 480
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    a = t.tensor(rows)
    return (a, tuple(a.shape), a.dtype == t.float32)


example = [[1, 2, 3], [4, 5, 6]]
print(solve(example))


<!-- dd:dd-q481 -->

### Problem 481 · independent

Write a function solve(values) that takes a FLAT Python list of numbers and returns a tuple (a, ndim). `a` is the tensor built from values, and `ndim` is its number of axes as a plain int. A flat list has one level of nesting, so think about what that makes ndim before you run it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([4, 1, 7]), 1)
```


In [ ]:
import torch
import torch as t

def solve(values):
    """Return (tensor built from values, its number of axes)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [4, 1, 7]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(481)


In [ ]:
#@title 💡 Solution — Problem 481
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return (tensor built from values, its number of axes)."""
    a = t.tensor(values)
    return (a, a.ndim)


example = [4, 1, 7]
print(solve(example))


<!-- dd:dd-q483 -->

### Problem 483 · independent

Write a function solve(values) that takes a FLAT Python list of numbers and returns a tuple (dtype_name, numel). `dtype_name` is str() of the tensor's dtype — the string 'torch.int64' or 'torch.float32'. `numel` is the element count as a plain int. A tensor holds ONE type for every element, so a single float anywhere in the list decides the type of the whole block.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
('torch.int64', 3)
```


In [ ]:
import torch
import torch as t

def solve(values):
    """Return (str of the tensor's dtype, its element count)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [1, 2, 3]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(483)


In [ ]:
#@title 💡 Solution — Problem 483
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(values):
    """Return (str of the tensor's dtype, its element count)."""
    a = t.tensor(values)
    return (str(a.dtype), a.numel())


example = [1, 2, 3]
print(solve(example))


<!-- dd:dd-q485 -->

### Problem 485 · independent

Write a function solve(cube) that takes a TRIPLY nested Python list — a list of equal-length lists of equal-length lists of numbers — and returns a tuple (ndim, shape, numel). `shape` must be a plain Python tuple of ints, not a torch.Size. Nesting depth becomes the number of axes: the outermost list is axis 0.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(3, (2, 2, 2), 8)
```


In [ ]:
import torch
import torch as t

def solve(cube):
    """Return (number of axes, shape as a plain tuple, element count)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(485)


In [ ]:
#@title 💡 Solution — Problem 485
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(cube):
    """Return (number of axes, shape as a plain tuple, element count)."""
    a = t.tensor(cube)
    return (a.ndim, tuple(a.shape), a.numel())


example = [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
print(solve(example))


<!-- dd:dd-q486 -->

### Problem 486 · independent

Write a function solve(rows) that returns a tuple (inferred_name, forced_name, unchanged). Build one tensor from rows and let PyTorch infer the dtype; build a second from the same rows but force it to torch.float32 by passing dtype=. The first two entries are str() of each tensor's dtype; `unchanged` is True when the two dtypes are equal. Forcing is how you stop an all-integer input from silently giving you an integer tensor where the rest of your code expects floats.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
('torch.int64', 'torch.float32', False)
```


In [ ]:
import torch
import torch as t

def solve(rows):
    """Return (inferred dtype name, forced float32 dtype name, are they equal?)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = [[1, 2], [3, 4]]
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(486)


In [ ]:
#@title 💡 Solution — Problem 486
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(rows):
    """Return (inferred dtype name, forced float32 dtype name, are they equal?)."""
    inferred = t.tensor(rows)
    forced = t.tensor(rows, dtype=t.float32)
    return (str(inferred.dtype), str(forced.dtype), inferred.dtype == forced.dtype)


example = [[1, 2], [3, 4]]
print(solve(example))


#### Common mistakes

- **"A tensor is just a faster list."** — A list stores anything, a tensor
  stores exactly one dtype in one memory block. That's why `t.tensor([1, 2.5])`
  changes your integer to a float: PyTorch must pick ONE type for the block.
- **"I need to tell PyTorch the shape when converting data."** — `t.tensor`
  infers shape from the nesting. You only specify shapes with from-scratch
  constructors (`t.zeros((2, 3))`), covered next.
- **"shape is (columns, rows)."** — It is (axis 0, axis 1) = (rows, columns)
  for a matrix. Axis 0 is always the outermost nesting level.
- **"`ndim` and `numel()` are two names for the size."** — `ndim` counts axes,
  `numel()` counts elements. A 2×3 tensor has `ndim == 2` and `numel() == 6`.


<!-- dd:dd-kp-numpy-constructors -->

## Tensor constructors — zeros, ones, full, eye, *_like

`numpy.constructors`


### constructors and the shape argument


`t.tensor` converts data you already have. Just as often you need a tensor
**built from scratch** — a canvas of zeros to fill in, a mask of ones. PyTorch
has one constructor per pattern, and they all share the same calling
convention:

> **constructor(shape, dtype=...)** — say how big, optionally say what type.

- **`t.zeros(shape)`** — all entries `0.0`. The default "empty canvas".
- **`t.ones(shape)`** — all entries `1.0`.
- **`t.full(shape, v)`** — all entries equal to your value `v`.
- **`t.empty(shape)`** — allocates *without initializing* (contents are
  whatever bytes were in memory). Only worth it when you will overwrite
  every entry immediately.

Three of them side by side — same call shape, different fill:


In [ ]:
import torch as t

print(t.zeros((2, 3)))
print(t.ones((2, 3)))
print(t.full((2, 3), 7.0))




PyTorch accepts the shape **either way**: `t.zeros(2, 3)` and `t.zeros((2, 3))`
both give a 2×3 tensor. That is worth noticing precisely because NumPy does
*not* allow it — `np.zeros(2, 3)` is a `TypeError`, since NumPy reads the
second positional argument as the dtype. Code translated from NumPy will use
the tuple form, and it keeps working.


In [ ]:
loose = t.zeros(3, 4)
tupled = t.zeros((3, 4))
assert loose.shape == tupled.shape == (3, 4)
print(loose.shape, "==", tupled.shape)


Build a 3×4 canvas of zeros:


In [ ]:
import torch as t

# Both spellings mean the same thing in PyTorch.
board = t.zeros((3, 4))
same = t.zeros(3, 4)
assert board.shape == (3, 4) == same.shape
print(board)
print("both spellings give", tuple(board.shape), "and", tuple(same.shape))




Why: shape comes first and everything else is keyword-only, so the constructor
never has to guess whether you meant a dimension or a dtype.


<!-- dd:dd-q227 -->

### Problem 227 · faded — your turn

All-zeros float vector of a given length (must also work for length 0).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0., 0., 0., 0.])
```


In [ ]:
import torch as t

def solve(n):
    """Return a 1-D float tensor of n zeros."""
    return t._____(n)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(227)


In [ ]:
#@title 💡 Solution — Problem 227
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.zeros(n)


example = 4
print(solve(example))


### the dtype is float32 unless you say otherwise


**The default floating dtype is `torch.float32`,** even though the values print
like integers. This is a real difference from NumPy, whose default is
`float64` — the same line of code gives you half the precision here, which is
deliberate, because neural networks are trained in 32-bit.


In [ ]:
import torch as t

default = t.ones(3)
print(default, default.dtype)
assert default.dtype == t.float32




The values printed as `1.` with a trailing dot, and that dot is the whole
warning: they are floats, not the integers they look like.

Pass `dtype=` to override: `t.ones((2, 2), dtype=t.bool)` is a matrix of `True`
(1 as a boolean is `True`); `t.zeros(4, dtype=t.int64)` is integer zeros.
Checking `dtype` right after construction is the habit that catches the
float-by-default surprise before it propagates.


In [ ]:
flags = t.ones((2, 2), dtype=t.bool)
counts = t.zeros(4, dtype=t.int64)
print(flags)
print(counts, counts.dtype)
assert bool(flags.all()) and flags.dtype == t.bool


An all-`True` boolean mask — ones, with the dtype said out loud:


In [ ]:
import torch as t

# ones gives every entry the value 1 — and 1 as a boolean is True.
mask = t.ones((3, 4), dtype=t.bool)
assert bool(mask.all()) and mask.dtype == t.bool
print(mask)

# The default, for contrast — float32, not float64 and not int.
assert t.ones(3).dtype == t.float32
print("asked for bool:", mask.dtype, "| default:", t.ones(3).dtype)




Why: without `dtype=t.bool` this would be a float tensor of 1.0s that merely
*prints* like what you wanted — say the type when it matters.


<!-- dd:dd-q212 -->

### Problem 212 · faded — your turn

A rows×cols tensor where every entry is the boolean `True`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[True, True, True],
        [True, True, True]])
```


In [ ]:
import torch as t

def solve(rows, cols):
    """All-True boolean matrix of shape (rows, cols)."""
    return t.ones((rows, cols), dtype=_____)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(212)


In [ ]:
#@title 💡 Solution — Problem 212
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols):
    return t.ones((rows, cols), dtype=t.bool)


print(solve(2, 3))


### *_like — copy shape AND dtype from an existing tensor


The **`*_like` variants** (`t.zeros_like(x)`, `t.ones_like(x)`,
`t.full_like(x, v)`) copy both the shape *and the dtype* from an existing
tensor — the right tool whenever the question is "give me a blank tensor
shaped like this one". Reaching for `*_like` is both shorter and safer than
reading off `.shape` and `.dtype` yourself, and in real model code it also
carries across the device the original lives on.


In [ ]:
import torch as t

x = t.tensor([[3, -1, 4], [1, 5, -9]], dtype=t.int32)
print("zeros_like:", t.zeros_like(x).dtype)
print("zeros(x.shape):", t.zeros(x.shape).dtype)




Same shape from both, but only one of them still knows the tensor was
integer. Every `*_like` behaves that way:


In [ ]:
print(t.ones_like(x))
print(t.full_like(x, 7))
assert t.full_like(x, 7).dtype == x.dtype == t.int32


In [ ]:
import torch as t

# "Blank tensor shaped like x" — zeros_like copies shape AND dtype,
# so an int32 input yields an int32 result, not the float default.
x = t.tensor([[3, -1, 4], [1, 5, -9]], dtype=t.int32)
blank = t.zeros_like(x)
assert blank.shape == x.shape
assert blank.dtype == t.int32
print(blank)
print("copied dtype:", blank.dtype, "| t.zeros(x.shape) would give:",
      t.zeros(x.shape).dtype)




Why: `t.zeros(x.shape)` would lose the dtype (float default) — `_like`
keeps both properties in one call.


<!-- dd:dd-q41 -->

### Problem 41 · faded — your turn

Blank tensor matching BOTH the shape and dtype of an existing tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 0, 0],
        [0, 0, 0]], dtype=torch.int32)
```


In [ ]:
import torch as t

def solve(x):
    """Return an all-zeros tensor with x's shape and x's dtype."""
    return t._____(x)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(41)


In [ ]:
#@title 💡 Solution — Problem 41
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.zeros_like(x)


example = t.tensor([[3, -1, 4], [1, 5, -9]], dtype=t.int32)
print(solve(example))


### t.eye — the identity matrix


**`t.eye(n)`** is the n×n identity matrix: `1.0` on the main diagonal,
`0.0` elsewhere. It's the seed for anything diagonal-shaped: `v * t.eye(n)`
puts a constant v on the diagonal, and indexing its rows with a permutation
turns it into a permutation matrix (a trick you will use in the random-number
KP).


In [ ]:
import torch as t

print(t.eye(3))
print(5.0 * t.eye(3))




The permutation trick, since it is the least obvious one — reordering the
identity's ROWS builds the matrix that reorders a vector the same way:


In [ ]:
order = t.tensor([2, 0, 1])
P = t.eye(3)[order]
v = t.tensor([10.0, 20.0, 30.0])
print(P)
print(P @ v)
assert (P @ v).tolist() == [30.0, 10.0, 20.0]


In [ ]:
import torch as t

I = t.eye(3)
assert I.tolist() == [[1.0, 0.0, 0.0],
                      [0.0, 1.0, 0.0],
                      [0.0, 0.0, 1.0]]
print(I)




Why: the identity is a constructor, not something you assemble by loop —
and scaling it (`v * t.eye(n)`) is the one-liner for "v on the diagonal".


<!-- dd:dd-q228 -->

### Problem 228 · faded — your turn

The n×n identity matrix.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
```


In [ ]:
import torch as t

def solve(n):
    """n-by-n identity matrix."""
    return t._____(n)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(228)


In [ ]:
#@title 💡 Solution — Problem 228
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.eye(n)


example = 3
print(solve(example))


<!-- dd:dd-q48 -->

### Problem 48 · guided

Write a function solve(v, fill) that takes a 1-D PyTorch integer tensor v and an integer fill value. It should return a tensor of the same length in which every element at an odd index (indices 1, 3, 5, ...) has been replaced by fill, while every element at an even index keeps its original value. Do not use Python loops, and leave the caller's tensor v unmodified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 3, -1,  2, -1,  5, -1])
```


<details>
<summary>Hints</summary>

1. Nothing is being constructed from scratch here — you need a COPY of v
   that you are allowed to write into.
2. Odd indices are a slice with a step, and assigning a scalar into a
   slice broadcasts it across every selected position.
3. `out = v.clone()` then `out[1::2] = fill`. Skipping the clone mutates
   the caller's tensor, which the drill checks for.

</details>


In [ ]:
import torch as t

def solve(v, fill):
    """Return a copy of v with every odd index replaced by fill."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([3, 8, 2, 7, 5, 1])
print(solve(example, -1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(48)


In [ ]:
#@title 💡 Solution — Problem 48
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v, fill):
    out = v.clone()
    out[1::2] = fill
    return out


example = t.tensor([3, 8, 2, 7, 5, 1])
print(solve(example, -1))


<!-- dd:dd-q225 -->

### Problem 225 · independent

Write a function solve(n) that takes a non-negative integer n and returns a 1-D PyTorch tensor of length n in which every entry is 1.0. The result must use torch's default floating-point dtype, not integers. Note that torch's default float is float32, where numpy's is float64 — the grader checks the dtype as well as the values.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 1., 1., 1.])
```


In [ ]:
import torch as t

def solve(n):
    """Return a 1-D float tensor of length n where every entry is 1.0."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = 4
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(225)


In [ ]:
#@title 💡 Solution — Problem 225
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.ones(n)


example = 4
print(solve(example))


<!-- dd:dd-q50 -->

### Problem 50 · independent

Write a function solve(n, v) that takes an integer n >= 1 and a number v, and returns an n x n floating-point PyTorch tensor with v at every position on the main diagonal and 0.0 everywhere else.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2., 0., 0., 0.],
        [0., 2., 0., 0.],
        [0., 0., 2., 0.],
        [0., 0., 0., 2.]])
```


In [ ]:
import torch as t

def solve(n, v):
    """Return an n x n float tensor with v on the main diagonal, 0.0 elsewhere."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(4, 2.0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(50)


In [ ]:
#@title 💡 Solution — Problem 50
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, v):
    return t.eye(n) * v


print(solve(4, 2.0))


<!-- dd:dd-q213 -->

### Problem 213 · independent

Write a function solve(n, idx) that takes a length n and a valid index idx, and returns a 1-D float PyTorch tensor of n zeros except for a 1.0 at position idx — a standard basis (one-hot) vector.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])
```


In [ ]:
import torch as t

def solve(n, idx):
    """Return a length-n float tensor that is 1.0 at idx and 0.0 elsewhere."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(10, 4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(213)


In [ ]:
#@title 💡 Solution — Problem 213
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, idx):
    z = t.zeros(n)
    z[idx] = 1.0
    return z


print(solve(10, 4))


#### Common mistakes

- **"Constructors give me integers if I write `t.ones(5)`."** — The default
  dtype is `torch.float32` regardless of how the values look. If the grader (or
  your model) needs ints or bools, say so with `dtype=`.
- **"float is float — precision doesn't change when I port NumPy code."** —
  `np.ones(4)` is float64 and `t.ones(4)` is float32. Exact equality checks
  written against NumPy output can fail here for no reason other than the
  narrower dtype.
- **"`dtype=bool` works, like in NumPy."** — PyTorch wants its own dtype
  objects: `t.bool`, `t.int64`, `t.float32`. Python's builtin `bool` and `int`
  are accepted in some places but `t.*` is the spelling to learn.
- **"`t.empty` means a tensor with no elements."** — It means *uninitialized
  memory* of the full requested shape: garbage values, not zeros, not empty.
  Use `t.zeros` unless you will overwrite everything.


<!-- dd:dd-kp-numpy-slicing-views -->

## Slicing, views, and slice assignment

`numpy.slicing-views`


Slicing is how you name a rectangular piece of a tensor. The syntax
generalizes Python's list slicing in two ways, and adds one semantic twist
that trips everyone at least once.

**Syntax.** A slice is `start:stop:step` (stop exclusive, any part omittable),
and a multi-dimensional tensor takes **one slice per axis, separated by
commas** inside a single pair of brackets:

- `x[2:5]` — elements 2, 3, 4 of a vector.
- `z[0, :]` — row 0, all columns. `z[:, -1]` — every row, last column.
- `x[::2]` — every second element.

Negative *indices* count from the end (`-1` is the last element), exactly as in
Python.


In [ ]:
import torch as t

x = t.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
print("x        ", x)
print("x[2:5]   ", x[2:5])
print("x[::2]   ", x[::2])
print("x[-3:]   ", x[-3:])




One slice per axis, comma-separated — and note what an *int* in a slot does
that a slice does not: it removes that axis.


In [ ]:
z = t.tensor([[0, 1, 2, 3],
              [4, 5, 6, 7],
              [8, 9, 10, 11]])
print(z)
print("z[0, :]  ", z[0, :],  " shape", z[0, :].shape)     # int -> 1-D
print("z[:, -1] ", z[:, -1], " shape", z[:, -1].shape)
print("z[0:1, :]", z[0:1, :]," shape", z[0:1, :].shape)   # slice -> stays 2-D




**Negative *steps* are the exception.** NumPy reverses an axis with `x[::-1]`;
PyTorch refuses — it raises `ValueError: step must be greater than zero`. This
is probably the single most common surprise when moving NumPy habits to torch.
Reversal has its own function:

- **`t.flip(x, [0])`** — reverse along axis 0. `t.flip(z, [1])` mirrors each
  row left-right; `t.flip(z, [0])` reverses the row order (mirror top-bottom).
- **`t.rot90(z)`** — rotate 90° counterclockwise, the composition of a
  transpose and a flip.


In [ ]:
try:
    x[::-1]
except ValueError as err:
    print("ValueError:", err)

print("flip axis 0", t.flip(x, [0]))
print("mirror rows left-right")
print(t.flip(z, [1]))
print("rot90")
print(t.rot90(z))
assert t.equal(t.rot90(z), t.flip(z.T, [0]))




**The twist: slices are *views*, not copies.** A slice doesn't copy data — it
is a new window onto the *same* memory block. Two consequences:

1. **Writing through a slice writes the original.** That enables the single
   most useful idiom in this KP, **slice assignment**:
   `x[start:stop] = value` sets a whole range at once (the scalar is
   broadcast to every selected position — no loop).
2. **"Return a new tensor" tasks need an explicit `.clone()`** if you would
   otherwise be returning or mutating a view of the caller's data. Rule of
   thumb: mutate → `.clone()` first, unless the task says to modify in place.

`t.flip` is not in that category: it always returns a **copy**, so writing into
its result never touches the input.


In [ ]:
data = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
window = data[1:3]
window[0] = 99.0                 # window[0] IS data[1]
print("data after writing through the view:", data)
assert data[1].item() == 99.0

safe = data[1:3].clone()
safe[0] = -1.0
print("data after writing through the clone:", data)
assert data[1].item() == 99.0




Slice assignment is the same fact used deliberately — a whole range set at
once, the scalar broadcast across every selected position, no loop:


In [ ]:
y = t.zeros(6)
y[1:4] = 5.0
y[::2] = -1.0
print(y)
assert y.tolist() == [-1.0, 5.0, -1.0, 5.0, -1.0, 0.0]


Task: given a vector, produce a reversed copy; then blank out the middle of
another vector in place.


In [ ]:
import torch as t

x = t.tensor([1.0, 2.0, 3.0, 4.0])

# The NumPy reflex does not work here.
try:
    x[::-1]
    raised = False
except ValueError:
    raised = True
assert raised, "torch rejects negative slice steps"

print("x[::-1] raised ValueError:", raised)

# Reverse with flip instead — and flip hands back a COPY.
rev = t.flip(x, [0])
assert rev.tolist() == [4.0, 3.0, 2.0, 1.0]
rev[0] = 99.0                    # writes only into rev
assert x.tolist() == [1.0, 2.0, 3.0, 4.0]
print("wrote 99 into the flipped COPY:", rev, "-> x is still", x)

# A plain slice, by contrast, IS a view: writing through it writes x.
window = x[1:3]
window[0] = 99.0                 # window[0] is x[1]!
assert x[1] == 99.0
print("wrote 99 through a VIEW:       ", window, "-> x is now  ", x)
x[1] = 2.0                       # undo

# Slice ASSIGNMENT: set positions 1..3 (stop 4 exclusive) to 0 — in place,
# no loop. The scalar 0.0 is broadcast across the selected range.
y = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
y[1:4] = 0.0
assert y.tolist() == [1.0, 0.0, 0.0, 0.0, 5.0, 6.0]
print("after y[1:4] = 0:              ", y)




Why each step:

1. The failed `x[::-1]` is worth writing once deliberately. It is the fastest
   way to stop reaching for it by reflex later.
2. `t.flip(x, dims)` names the axes to reverse as a list — you pick *which*
   axis by what you put in that list, the same choice you would have made by
   which comma slot got the `::-1`.
3. The view demonstration is the mental model to keep: a slice is a window,
   not a photocopy. Cheap to make, dangerous to mutate casually.
4. Slice assignment replaces the `for i in range(start, stop)` loop entirely —
   and it is the building block for border/checkerboard/striping patterns in
   the next lesson.


<!-- dd:dd-q233 -->

### Problem 233 · faded — your turn

Reversed copy of a 1-D tensor (input unmodified).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([4., 3., 2., 1.])
```


In [ ]:
import torch as t

def solve(x):
    """Return a new tensor with x's elements in reverse order."""
    return t._____(x, [0])


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(233)


In [ ]:
#@title 💡 Solution — Problem 233
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.flip(x, [0])


example = t.tensor([1.0, 2.0, 3.0, 4.0])
print(solve(example))


<!-- dd:dd-q76 -->

### Problem 76 · guided

Write a function solve(z) that takes a 2-D PyTorch tensor and returns a tuple of two new tensors: first, z mirrored left-right (each row reversed); second, z mirrored top-bottom (the order of the rows reversed). PyTorch does not support negative-step slicing, so reach for the flip operation and pick the right axis for each. The input must not be modified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([[2, 1, 0],
        [5, 4, 3]]), tensor([[3, 4, 5],
        [0, 1, 2]]))
```


<details>
<summary>Hints</summary>

1. Two mirrors of a 2-D tensor: left-right (reverse within each row) and
   top-bottom (reverse the order of rows). Both are single `t.flip` calls.
2. `t.flip` takes the axes to reverse as a list. Which axis number reverses
   each row? Which reverses the row order?
3. `t.flip(z, [1])` and `t.flip(z, [0])` — and because flip copies, the
   "input must not be modified" requirement is already satisfied.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return (z mirrored left-right, z mirrored top-bottom)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(6).reshape(2, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(76)


In [ ]:
#@title 💡 Solution — Problem 76
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.flip(z, [1]), t.flip(z, [0])


example = t.arange(6).reshape(2, 3)
print(solve(example))


<!-- dd:dd-q506 -->

### Problem 506 · guided

Write a function solve(x, k) that takes a 1-D tensor and an int, and returns the first k elements. A slice with no start means 'from the beginning'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 2, 3])
```


<details>
<summary>Hints</summary>

1. A slice, not an index — you want a run of elements, not one element.
2. Leaving the start empty means 'from the beginning'.
3. `x[:k]`.

</details>


In [ ]:
import torch as t

def solve(x, k):
    """Return the first k elements."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1, 2, 3, 4, 5]), 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(506)


In [ ]:
#@title 💡 Solution — Problem 506
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, k):
    """Return the first k elements."""
    return x[:k]


example = (t.tensor([1, 2, 3, 4, 5]), 3)
print(solve(*example))


<!-- dd:dd-q231 -->

### Problem 231 · independent

Write a function solve(x, start, stop, value) that takes a 1-D PyTorch tensor of floats x, two integer indices start and stop, and a float value. Set every entry of x from index start up to (but not including) index stop to value, leave every entry outside that range unchanged, and return the tensor. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 0., 0., 0., 5., 6.])
```


In [ ]:
import torch as t

def solve(x, start, stop, value):
    """Set x[start:stop] to value in place and return x."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
print(solve(example, 1, 4, 0.0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(231)


In [ ]:
#@title 💡 Solution — Problem 231
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, start, stop, value):
    x[start:stop] = value
    return x


example = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
print(solve(example, 1, 4, 0.0))


<!-- dd:dd-q75 -->

### Problem 75 · independent

Write a function solve(z) that takes a 2-D PyTorch tensor and returns it rotated 90 degrees counterclockwise: the last column of the input becomes the first row of the output. An input of shape (r, c) produces output of shape (c, r).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2, 5],
        [1, 4],
        [0, 3]])
```


In [ ]:
import torch as t

def solve(z):
    """Rotate the 2-D tensor z by 90 degrees counterclockwise."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(6).reshape(2, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(75)


In [ ]:
#@title 💡 Solution — Problem 75
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.rot90(z)


example = t.arange(6).reshape(2, 3)
print(solve(example))


<!-- dd:dd-q507 -->

### Problem 507 · independent

Write a function solve(x, col) that returns a tuple (values, shape) for column `col` of the 2-D tensor x: the column's values as a plain list, and its shape as a plain tuple. Indexing an axis with a plain int REMOVES that axis, so the result is 1-D, not a column-shaped 2-D tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([2, 5], (2,))
```


In [ ]:
import torch as t

def solve(x, col):
    """Return one column of a 2-D tensor, and its shape."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1, 2, 3], [4, 5, 6]]), 1)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(507)


In [ ]:
#@title 💡 Solution — Problem 507
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, col):
    """Return one column of a 2-D tensor, and its shape."""
    c = x[:, col]
    return (c.tolist(), tuple(c.shape))


example = (t.tensor([[1, 2, 3], [4, 5, 6]]), 1)
print(solve(*example))


<!-- dd:dd-q74 -->

### Problem 74 · independent

Write a function solve(z, step, v) that takes a 1-D PyTorch tensor z, a positive integer step, and a value v. Return a new tensor equal to z except that every step-th element — indices 0, step, 2*step, ... — has been overwritten with v. The input tensor must not be modified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([-1,  1,  2, -1,  4,  5, -1,  7,  8, -1, 10, 11])
```


In [ ]:
import torch as t

def solve(z, step, v):
    """Return a copy of z with every step-th element replaced by v."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(12)
print(solve(example, 3, -1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(74)


In [ ]:
#@title 💡 Solution — Problem 74
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, step, v):
    out = z.clone()
    out[::step] = v
    return out


example = t.arange(12)
print(solve(example, 3, -1))


<!-- dd:dd-q189 -->

### Problem 189 · independent

Write a function solve(z, k) that takes a 2-D PyTorch tensor and an integer k (possibly 0 or larger than 3), and returns z rotated counterclockwise by k quarter-turns (k=1 is 90 degrees CCW; k is taken modulo 4). Do not modify the input.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2, 5],
        [1, 4],
        [0, 3]])
```


In [ ]:
import torch as t

def solve(z, k):
    """Rotate z counterclockwise by k quarter-turns."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.arange(6).reshape(2, 3), 1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(189)


In [ ]:
#@title 💡 Solution — Problem 189
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, k):
    return t.rot90(z, k)


print(solve(t.arange(6).reshape(2, 3), 1))


#### Common mistakes

- **"`x[::-1]` reverses a tensor."** — It raises `ValueError: step must be
  greater than zero`. PyTorch supports no negative slice steps at all;
  `t.flip(x, [0])` is the operation.
- **"A slice is a copy."** — It's a view of the same memory. Mutating a slice
  mutates the original. When a task says "return a new tensor" or "do not
  modify the input" and you plan to write into the result, `.clone()` first.
- **"`.copy()` makes the copy."** — That's the NumPy name. Tensors clone with
  `.clone()`.
- **"2-D indexing is `z[i][j]`."** — That works but chains two operations;
  the idiom is one bracket, comma-separated: `z[i, j]`, `z[i, :]`, `z[:, j]`.
  The chained form also breaks down for slice-then-assign patterns.


<!-- dd:dd-kp-numpy-ranges -->

## Numeric ranges — arange and linspace

`numpy.ranges`


### t.arange — the stop is exclusive


When you know the **step**, use **`t.arange(start, stop, step)`** (step
defaults to 1). It counts from `start` in increments of `step` and — exactly
like Python's `range` — **stops BEFORE `stop`**. The endpoint is never
included, even with a step: `t.arange(0, 10, 2)` is `[0, 2, 4, 6, 8]` — 10
is left out.


In [ ]:
import torch as t

print(t.arange(5))
print(t.arange(0, 10, 2))          # 10 is the stop, so 10 is missing
print(t.arange(0, 11, 2))          # push the stop past it to get it back




That exclusive stop is the whole trick, and the source of most range bugs.
When a task wants the endpoint *included*, you have to extend the stop past
where you want to end.

`t.arange` over integers gives you an **integer** tensor (`int64`), which
matters because integer tensors are what you index with.


In [ ]:
idx = t.arange(3)
letters = t.tensor([10, 20, 30, 40])
print(idx.dtype, "->", letters[idx])
assert idx.dtype == t.int64


In [ ]:
import torch as t

# Counting by 2 up to 10 — but 10 is the stop, so it's EXCLUDED.
evens = t.arange(0, 10, 2)
assert evens.tolist() == [0, 2, 4, 6, 8]
assert evens.dtype == t.int64
print(evens, evens.dtype, "  <- no 10")




Why: notice 10 never appears. The step doesn't change the rule — `arange`
always halts one step short of `stop`.


<!-- dd:dd-q229 -->

### Problem 229 · faded — your turn

Every integer from `start` to `end`, **including both endpoints**. (Watch the
exclusive stop — how do you make `end` appear?)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([3, 4, 5, 6, 7, 8])
```


In [ ]:
import torch as t

def solve(start, end):
    """Integers start..end inclusive, in order."""
    return t.arange(start, _____)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(229)


In [ ]:
#@title 💡 Solution — Problem 229
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(start, end):
    return t.arange(start, end + 1)


example_start, example_end = 3, 8
print(solve(example_start, example_end))


### t.linspace — you know the number of points


When you know the **number of points** instead of the step, use
**`t.linspace(start, stop, num)`**: exactly `num` evenly spaced values, and
this time **both endpoints are included**. Mind the fencepost — `num` points
make `num − 1` gaps, so `t.linspace(0.0, 1.0, 5)` has step 1/4, not 1/5.


In [ ]:
import torch as t

grid = t.linspace(0.0, 1.0, 5)
print(grid)
print("gaps:", grid[1:] - grid[:-1])   # 5 points, so 4 of them
assert grid[0].item() == 0.0 and grid[-1].item() == 1.0




Both ends present, four gaps between five points. Side by side with `arange`
the two conventions are hard to confuse again:


In [ ]:
print("arange  ", t.arange(0.0, 1.0, 0.25))     # stop excluded -> 4 values
print("linspace", t.linspace(0.0, 1.0, 5))      # stop included -> 5 values
assert len(t.arange(0.0, 1.0, 0.25)) == 4
assert len(t.linspace(0.0, 1.0, 5)) == 5


In [ ]:
import torch as t

# 5 points from 0 to 1, endpoints INCLUDED -> 4 equal gaps of 0.25.
grid = t.linspace(0.0, 1.0, 5)
assert grid.tolist() == [0.0, 0.25, 0.5, 0.75, 1.0]
print(grid, " 5 points,", len(grid) - 1, "gaps")




Why: both 0.0 and 1.0 are present — that's the opposite of `arange`. Count
the points, not the intervals.


<!-- dd:dd-q242 -->

### Problem 242 · faded — your turn

The `n` evenly spaced breakpoints **strictly inside** (0, 1) — exclude 0.0 and
1.0. (linspace includes the endpoints; how do you get `n` points *between*
them?)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.2500, 0.5000, 0.7500])
```


In [ ]:
import torch as t

def solve(n):
    """n interior breakpoints of (0, 1), endpoints excluded."""
    return t.linspace(0.0, 1.0, n + _____)[1:-1]


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(242)


In [ ]:
#@title 💡 Solution — Problem 242
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.linspace(0.0, 1.0, n + 2)[1:-1]


example = 3
print(solve(example))


### float steps drift — count, then scale


`t.arange` with a **float** step is a trap: each element is built by repeated
addition, so rounding error accumulates and the endpoint may or may not show
up. It is a sharper trap here than in NumPy, because the default float is
32-bit and has fewer digits to lose.

The robust recipe is to turn the step-question into a count-question:
figure out how many points there are, generate exact **integers**, and scale
them once — `t.arange(n_points) * step`. One multiply per element, no drift.


In [ ]:
import torch as t

drifty = t.arange(0.0, 1.0 + 0.1, 0.1)
print(drifty)
print("last value:", drifty[-1].item(), "| point count:", len(drifty))




The last entry is not the clean `1.0` the call asked for, and whether an
eleventh point appears at all is decided by rounding error. Counting first
removes the gamble:


In [ ]:
n = int(round(1.0 / 0.1)) + 1
exact = t.arange(n) * 0.1
print(exact)
print("point count:", len(exact))
assert len(exact) == 11




The count uses the exclusive-stop insight again: an inclusive range of
`step`-spaced points from 0 to `stop` has `round(stop / step) + 1` of them.


In [ ]:
import torch as t

# 0 to 1 inclusive, spacing 0.25. Count the points, scale integers.
n = int(round(1.0 / 0.25)) + 1        # 5 points: 0, 0.25, 0.5, 0.75, 1.0
grid = t.arange(n) * 0.25
assert grid.tolist() == [0.0, 0.25, 0.5, 0.75, 1.0]
print("n =", n, "->", grid)




Why: `t.arange(0, 1.0 + 0.25, 0.25)` would gamble on the endpoint;
`t.arange(n) * step` is exact because the integers are exact.


<!-- dd:dd-q214 -->

### Problem 214 · faded — your turn

`solve(stop, step)`: the inclusive float range 0, step, 2·step, …, up to **and
including** `stop` (an exact multiple of `step`). (Why does the point count
need a `+ 1`?)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 0.0000,  0.5000,  1.0000,  1.5000,  2.0000,  2.5000,  3.0000,  3.5000,
         4.0000,  4.5000,  5.0000,  5.5000,  6.0000,  6.5000,  7.0000,  7.5000,
         8.0000,  8.5000,  9.0000,  9.5000, 10.0000])
```


In [ ]:
import torch as t

def solve(stop, step):
    """0, step, ..., stop inclusive — exact, no float drift."""
    n = int(round(stop / step)) + _____
    return t.arange(n) * step


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(214)


In [ ]:
#@title 💡 Solution — Problem 214
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(stop, step):
    n = int(round(stop / step)) + 1
    return t.arange(n) * step


print(solve(10.0, 0.5))


<!-- dd:dd-q524 -->

### Problem 524 · guided

Write a function solve(high, low) that takes two integers with high >= low and returns a 1-D integer tensor counting DOWN from high to low with BOTH endpoints included: high, high - 1, ..., low. Use a single t.arange call with a negative step. The stop is exclusive when counting down too, so decide which side of low the stop has to sit on.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([5, 4, 3, 2, 1])
```


<details>
<summary>Hints</summary>

1. The exclusive stop does not care which way you are counting. Going down,
   "stops before the stop" means the stop has to sit one step PAST `low`.
2. A negative step reverses the direction: `t.arange(high, ?, -1)`. Ask what
   `?` makes `low` the last value actually produced.
3. `t.arange(high, low - 1, -1)` — the `- 1` is the inclusive-endpoint fix
   from q229, pointed the other way.

</details>


In [ ]:
import torch
import torch as t

def solve(high, low):
    """Integers from high down to low, both endpoints included."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (5, 1)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(524)


In [ ]:
#@title 💡 Solution — Problem 524
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(high, low):
    """Integers from high down to low, both endpoints included."""
    return t.arange(high, low - 1, -1)


example = (5, 1)
print(solve(*example))


<!-- dd:dd-q53 -->

### Problem 53 · independent

Write a function solve(n) that takes an integer n >= 2 and returns a 1-D floating-point PyTorch tensor of n evenly spaced values that starts at exactly 0.0 and ends at exactly 1.0, with equal spacing between consecutive entries.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
```


In [ ]:
import torch as t

def solve(n):
    """Return n evenly spaced floats from 0.0 to 1.0 inclusive."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(53)


In [ ]:
#@title 💡 Solution — Problem 53
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.linspace(0.0, 1.0, n)


print(solve(5))


#### Common mistakes

- **"`t.arange(3, 8)` includes 8."** — Like Python's `range`, the stop is
  exclusive. Endpoint bugs from this are the most common range mistake; when
  a task says "inclusive", plan the `+ step` (or switch to `linspace`).
- **"Float steps in arange are fine."** — Each element is built by repeated
  float addition, so the endpoint may or may not appear and interior values
  drift. Scale exact integers (`t.arange(n) * step`) or use `linspace`.
- **"`linspace(0, 1, 5)` has step 1/5."** — It has step 1/4: five points means
  FOUR gaps. `linspace` counts points, not intervals.
- **"`t.arange(5)` gives floats."** — Integer arguments give an `int64` tensor.
  That is what you want for indexing; if you need floats, ask for them
  (`t.arange(5.0)` or `dtype=t.float32`).


<!-- dd:dd-kp-numpy-dtype-astype -->

## Dtypes, .to(), and memory size

`numpy.dtype-astype`


### .to() — converting an existing tensor


Every tensor has exactly one **dtype** — the type shared by all of its
elements. It determines what the values can be: `int64` can't hold 0.5;
`float32` holds it with less precision than `float64`; `bool` holds only
`True`/`False`. Mixed inputs get promoted to the common type (ints + one
float → all float), silently.

To convert an existing tensor: **`x.to(new_dtype)`** (this is PyTorch's
`astype`). It returns a *new* tensor when the dtype actually changes — the
original is untouched; there is no in-place dtype change. Converting
float→int **truncates toward zero** rather than rounding: `1.9 → 1`,
`-1.9 → -1`.


In [ ]:
import torch as t

x = t.tensor([1, 2, 3])
y = x.to(t.float32)
print(x, x.dtype)
print(y, y.dtype)
assert x.dtype == t.int64          # the original never changed




Truncation is the part that surprises people — it is not rounding:


In [ ]:
print(t.tensor([1.9, -1.9, 0.5]).to(t.int64))
assert t.tensor([1.9]).to(t.int64).item() == 1
assert t.tensor([-1.9]).to(t.int64).item() == -1




One wrinkle worth knowing: if the dtype you ask for is the one it already
has, `.to()` hands back the *same* tensor rather than a copy. It promises a
tensor of that dtype, not a fresh buffer.


In [ ]:
same = x.to(t.int64)
print("same object?", same is x)
assert same is x


Make a float32 copy of an integer tensor:


In [ ]:
import torch as t

x = t.tensor([1, 2, 3])
assert x.dtype == t.int64           # default integer dtype

# .to() returns a NEW tensor with converted values; x is untouched.
y = x.to(t.float32)
assert y.dtype == t.float32
assert x.dtype == t.int64           # original unchanged — .to() copies
print("x", x, x.dtype)
print("y", y, y.dtype)




Why: checking `x.dtype` first tells you what conversion is actually needed —
don't convert blind. And dtype is a property of the whole memory block, so
changing it means building a new block.


<!-- dd:dd-q230 -->

### Problem 230 · faded — your turn

Float32 copy of an integer tensor, original left unmodified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 2., 3.])
```


In [ ]:
import torch as t

def solve(x):
    """Return a float32 copy of integer tensor x."""
    return x._____(t.float32)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(230)


In [ ]:
#@title 💡 Solution — Problem 230
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.to(t.float32)


example = t.tensor([1, 2, 3])
print(solve(example))


### dtypes at creation, and dtype names as strings


Rather than build-then-convert, request the dtype at creation: every
constructor accepts `dtype=`, e.g. `t.arange(n, dtype=t.int32)` — one
step, no copy.


In [ ]:
import torch as t

built = t.arange(4, dtype=t.int32)
print(built, built.dtype)




Here PyTorch is stricter than NumPy. NumPy accepts the *string* `'float32'`
anywhere a dtype is wanted; PyTorch does not — `t.arange(3, dtype='float32')`
raises a `TypeError`. Watch it happen:


In [ ]:
try:
    t.arange(3, dtype='float32')
except TypeError as err:
    print("TypeError:", err)




When the dtype arrives as **data** (from a config, a file header, a function
argument), you have to turn the name into the dtype object first, and
`getattr(t, name)` does exactly that.


In [ ]:
for name in ('int32', 'float32', 'float64'):
    z = t.arange(3, dtype=getattr(t, name))
    print(f"{name:8} -> {z} {z.dtype}")
assert t.arange(3, dtype=getattr(t, 'float32')).dtype == t.float32


Build 0..2 already in float32 — dtype named by a string:


In [ ]:
import torch as t

name = 'float32'

# PyTorch wants the dtype OBJECT, so look it up from the name.
z = t.arange(3, dtype=getattr(t, name))
assert z.dtype == t.float32
print(repr(name), "->", getattr(t, name), "->", z)




Why: when a function receives `dtype_str` as an argument, one `getattr` turns
it into the real dtype — no lookup table from strings to torch objects, and no
`if/elif` chain over dtype names.


<!-- dd:dd-q215 -->

### Problem 215 · faded — your turn

The integers 0..n-1 stored with a dtype named by a string.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0, 1, 2, 3, 4], dtype=torch.int32)
```


In [ ]:
import torch as t

def solve(n, dtype_str):
    """0..n-1 with the dtype named by dtype_str (e.g. 'int32')."""
    return t.arange(n, dtype=_____(t, dtype_str))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(215)


In [ ]:
#@title 💡 Solution — Problem 215
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, dtype_str):
    return t.arange(n, dtype=getattr(t, dtype_str))


print(solve(5, "int32"))


### dtype is a memory choice — element_size and numel


The number in a dtype's name is bits: `int32` = 4 bytes per element,
`float64` = 8. Total buffer size is `elements × bytes-per-element` —
**`x.numel()`** elements at **`x.element_size()`** bytes each. Dtype choices
are memory choices: halving precision halves the buffer, which is the whole
reason models train in float32 (or bfloat16) rather than float64.


In [ ]:
import torch as t

grid = t.zeros((10, 10))
for dtype in (t.float64, t.float32, t.int16, t.bool):
    z = grid.to(dtype)
    print(f"{str(dtype):15} {z.numel()} x {z.element_size()} = "
          f"{z.numel() * z.element_size()} bytes")




Same 100 numbers, an 8× spread in what they cost:


In [ ]:
assert grid.to(t.float64).element_size() == 2 * grid.to(t.float32).element_size()
assert grid.to(t.float32).numel() * grid.to(t.float32).element_size() == 400
print("float64 is exactly", grid.to(t.float64).element_size(), "bytes/element,",
      "float32", grid.to(t.float32).element_size())
print("100 float32 elements =",
      grid.to(t.float32).numel() * grid.to(t.float32).element_size(), "bytes")


In [ ]:
import torch as t

x = t.tensor([1, 2, 3])             # int64: 8 bytes each
assert x.numel() * x.element_size() == 3 * 8

# float32 elements are 4 bytes, so the converted copy is half the size.
y = x.to(t.float32)
assert y.element_size() == 4
assert y.numel() * y.element_size() == 3 * 4
print(x.dtype, x.numel() * x.element_size(), "bytes")
print(y.dtype, y.numel() * y.element_size(), "bytes")




Why: `numel × element_size` makes the cost concrete — a 10×10 float32
tensor is exactly 400 bytes, where the float64 NumPy equivalent is 800.


<!-- dd:dd-q51 -->

### Problem 51 · faded — your turn

A tensor's buffer size, reported as the string "&lt;n&gt; bytes".

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
400 bytes
```


In [ ]:
import torch as t

def solve(z):
    """Return z's data-buffer size as e.g. '24 bytes'."""
    return f"{z.numel() * z._____()} bytes"


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(51)


In [ ]:
#@title 💡 Solution — Problem 51
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return f"{z.numel() * z.element_size()} bytes"


example = t.zeros((10, 10))
print(solve(example))


<!-- dd:dd-q87 -->

### Problem 87 · guided

Write a function solve(z, bins) that takes a 1-D PyTorch tensor of floats in [0, 1) and a positive integer bins, and returns a 1-D INTEGER tensor of length bins counting how many values fall into each of the bins equal-width intervals dividing [0, 1]. torch's histogram counter returns floats, so converting the dtype is part of the exercise.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 0, 2, 0, 1])
```


<details>
<summary>Hints</summary>

1. Torch has a histogram counter that takes the bin count and the range
   directly — no bucketing by hand.
2. It returns FLOAT counts. The drill wants integers, so the last step is
   a dtype conversion.
3. `t.histc(z, bins=bins, min=0.0, max=1.0).to(t.int64)` — this is the
   dtype lesson in miniature: the numbers were already right, only their
   type was wrong.

</details>


In [ ]:
import torch as t

def solve(z, bins):
    """Return integer counts of z's values across `bins` equal-width bins over [0, 1]."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([0.1, 0.5, 0.55, 0.9])
print(solve(example, 5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(87)


In [ ]:
#@title 💡 Solution — Problem 87
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, bins):
    counts = t.histc(z, bins=bins, min=0.0, max=1.0)
    return counts.to(t.int64)


example = t.tensor([0.1, 0.5, 0.55, 0.9])
print(solve(example, 5))


<!-- dd:dd-q19 -->

### Problem 19 · independent

Write a function solve(z) that takes a 1-D complex-valued PyTorch tensor and returns a tuple of two real-valued tensors of the same length: the first holding the real part of each entry, the second holding the imaginary part.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([1., 3.]), tensor([2., 4.]))
```


In [ ]:
import torch as t

def solve(z):
    """Split a complex tensor into its real and imaginary parts."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1 + 2j, 3 + 4j])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(19)


In [ ]:
#@title 💡 Solution — Problem 19
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.real, z.imag


example = t.tensor([1 + 2j, 3 + 4j])
print(solve(example))


#### Common mistakes

- **"`.to()` changes the tensor in place."** — It returns a converted copy;
  the original keeps its dtype and values. If you meant to keep it, assign it:
  `x = x.to(...)`.
- **"Float→int conversion rounds."** — It truncates toward zero: `1.9 → 1`,
  `-1.9 → -1`. If you want rounding, round first: `x.round().to(t.int64)`.
- **"`dtype='float32'` works, like in NumPy."** — PyTorch rejects the string
  with a `TypeError`. Pass `t.float32`, or `getattr(t, name)` when the name
  arrives as data.
- **"dtype names are just labels."** — The number is the bit width, which sets
  both the representable range/precision and the memory per element.
  `numel × element_size` — a 10×10 float32 tensor is exactly 400 bytes.


<!-- dd:dd-kp-numpy-reshape-flatten -->

## Reshape, flatten, and element order

`numpy.reshape-flatten`


A tensor's data is one flat block of memory; the shape is just metadata saying
how to read it. **Reshaping changes the metadata without touching the data** —
which is why it's usually free (no copy) and why the one hard rule is:

> the new shape's element count must equal the old one
> (`2 × 6 = 12 = 3 × 4` ✓, but 12 → `(5, 3)` ✗ raises an error).

The two directions:

- **`x.reshape(shape)`** — reinterpret the flat data as a new shape.
  A convenience worth memorizing: one dimension may be **`-1`**, meaning
  "compute this one for me": `x.reshape(3, -1)` figures out the columns.
- **`x.flatten()`** — collapse any shape back to 1-D.


In [ ]:
import torch as t

x = t.arange(12)
print(x.reshape(3, 4))
print("(2, 6) ", x.reshape(2, 6).shape)
print("(3, -1)", x.reshape(3, -1).shape)
print("flatten", x.reshape(3, 4).flatten())

try:
    x.reshape(5, 3)                      # 15 != 12
except RuntimeError as err:
    print("RuntimeError:", err)




The count rule is not a guideline — a mismatch raises rather than padding or
truncating.

You will also meet **`x.view(shape)`**, which is reshape's stricter sibling: it
*only* ever re-labels the existing memory and raises if that is impossible.
`reshape` falls back to copying in that case. Prefer `reshape` unless you
specifically want the error.

The question that makes reshape make sense is: **in what order do elements
fill the new shape?** PyTorch's order is **row-major ("C order")**: the
*last* axis varies fastest. Reading a 2-D tensor in row-major order means
walking across row 0 left to right, then row 1, and so on. So

```python no-run
t.arange(6).reshape(2, 3)   # → [[0, 1, 2],
                            #    [3, 4, 5]]
```

fills row 0 first. This single fact explains most reshape results, including
higher-dimensional ones: `reshape(2, 2, 3)` fills the last axis (length 3)
fastest, the first axis slowest.


In [ ]:
print(t.arange(6).reshape(2, 3))
print(t.arange(12).reshape(2, 2, 3))




Read the 3-D print bottom-up: the innermost brackets (length 3) count by one,
so that axis moves fastest; the outermost changes only once.

Column-major ("Fortran") order — first axis fastest — has **no keyword** in
PyTorch. NumPy spells it `order='F'`; here you get it by changing the axis
order first and then flattening: `z.T.flatten()` reads the matrix column by
column.


In [ ]:
z = t.arange(6).reshape(2, 3)
print(z)
print("row-major   ", z.flatten())
print("column-major", z.T.flatten())
assert z.T.flatten().tolist() == [0, 3, 1, 4, 2, 5]




Since reshape only re-labels memory, a reshaped tensor is often a **view** —
writing into it writes the original:


In [ ]:
base = t.zeros(6)
grid = base.reshape(2, 3)
grid[0, 0] = 9.0
print("base:", base)
assert base[0].item() == 9.0


Task: build the classic "counting matrix" — an n×n tensor containing 0..n²-1
reading left-to-right, top-to-bottom — then flatten it back both ways.


In [ ]:
import torch as t

n = 3
# Step 1: make the flat sequence 0..8. It's 1-D — shape (9,).
flat = t.arange(n * n)

# Step 2: reshape to (3, 3). Row-major fill means 0,1,2 land in row 0 —
# exactly the "reading order" the task describes. No data is copied.
grid = flat.reshape(n, n)
assert grid.tolist() == [[0, 1, 2], [3, 4, 5], [6, 7, 8]]

# The -1 shortcut: "3 rows, you work out the columns."
assert grid.tolist() == t.arange(9).reshape(3, -1).tolist()

# Step 3: flatten undoes it — row-major walk gives back 0..8 in order.
assert grid.flatten().tolist() == [0, 1, 2, 3, 4, 5, 6, 7, 8]

# Column-major walk reads DOWN each column instead. No order= keyword
# exists, so transpose first and let the row-major walk do the work.
assert grid.T.flatten().tolist() == [0, 3, 6, 1, 4, 7, 2, 5, 8]
print(grid)
print("row-major   ", grid.flatten())
print("column-major", grid.T.flatten())




Why each step:

1. `arange` + `reshape` is the standard two-step for "matrix containing the
   numbers 0..k in reading order" — generate the flat values, then organize
   them. It works because reshape's fill order IS reading order.
2. `-1` earns its keep when one dimension is derived: you state the part you
   know and let PyTorch check the arithmetic.
3. The two flattens show that "flatten" is not one operation until you say the
   order. The default matches how the matrix prints; the column-by-column read
   is a transpose away.


<!-- dd:dd-q46 -->

### Problem 46 · faded — your turn

n×n matrix containing 0..n²-1 in reading order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [3, 4, 5],
        [6, 7, 8]])
```


In [ ]:
import torch as t

def solve(n):
    """Return the n x n matrix of 0..n*n-1 in row-major reading order."""
    return t.arange(_____).reshape(_____, _____)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(46)


In [ ]:
#@title 💡 Solution — Problem 46
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n):
    return t.arange(n * n).reshape(n, n)


example = 3
print(solve(example))


<!-- dd:dd-q36 -->

### Problem 36 · guided

Write a function solve(a, shape) that takes a 1-D PyTorch tensor a and a tuple shape of three positive integers whose product equals the length of a. Return a 3-D tensor with exactly that shape, containing a's values in the same row-major order (the last axis varies fastest). The values themselves must be unchanged — only how they are organized into dimensions changes.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[ 0.,  1.,  2.],
         [ 3.,  4.,  5.]],

        [[ 6.,  7.,  8.],
         [ 9., 10., 11.]]])
```


<details>
<summary>Hints</summary>

1. You get a 1-D tensor and a target 3-D shape whose product equals its length —
   this is a pure reorganize-the-metadata task.
2. "Same row-major order (the last axis varies fastest)" in the prompt is
   describing reshape's DEFAULT fill order — no reordering needed on your part.
3. One method call on the input tensor does the whole job.

</details>


In [ ]:
import torch as t

def solve(a, shape):
    """Reshape the 1-D tensor a into the given 3-D shape."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(12.0)
print(solve(example, (2, 2, 3)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(36)


In [ ]:
#@title 💡 Solution — Problem 36
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, shape):
    return a.reshape(shape)


example = t.arange(12.0)
print(solve(example, (2, 2, 3)))


<!-- dd:dd-q490 -->

### Problem 490 · guided

Write a function solve(x) that takes a PyTorch tensor of any shape and returns a 1-D tensor holding the same values. Elements come out in row-major order: the LAST axis varies fastest.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 2, 3, 4, 5, 6])
```


<details>
<summary>Hints</summary>

1. Any shape in, exactly one axis out.
2. You are not choosing a new shape, you are removing all structure — so no
   dimensions need naming.
3. `x.flatten()`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return x collapsed to a single axis."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [4, 5, 6]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(490)


In [ ]:
#@title 💡 Solution — Problem 490
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return x collapsed to a single axis."""
    return x.flatten()


example = t.tensor([[1, 2, 3], [4, 5, 6]])
print(solve(example))


<!-- dd:dd-q491 -->

### Problem 491 · guided

Write a function solve(x, cols) that takes a tensor and a column count, and returns it reshaped into a 2-D tensor with exactly `cols` columns. Do not compute the row count yourself — pass -1 for it and let PyTorch infer the only value that can work.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [3, 4, 5]])
```


<details>
<summary>Hints</summary>

1. You know one of the two numbers in the target shape. You do not know the
   other, and you should not compute it.
2. Reshape accepts -1 in exactly one position, meaning 'work this out'.
3. `x.reshape(-1, cols)`.

</details>


In [ ]:
import torch as t

def solve(x, cols):
    """Return x reshaped to `cols` columns, inferring the row count."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.arange(6), 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(491)


In [ ]:
#@title 💡 Solution — Problem 491
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, cols):
    """Return x reshaped to `cols` columns, inferring the row count."""
    return x.reshape(-1, cols)


example = (t.arange(6), 3)
print(solve(*example))


<!-- dd:dd-q23 -->

### Problem 23 · independent

Write a function solve(z) that takes a 2-D PyTorch tensor and returns a 1-D tensor listing all of z's entries in column-major order: read down the first column top to bottom, then the second column, and so on. Note that this is not the default row-major flattening, and unlike numpy, torch's flatten has no order argument — so change the axis order first.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0, 3, 1, 4, 2, 5])
```


In [ ]:
import torch as t

def solve(z):
    """Flatten z in column-major order."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(6).reshape(2, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(23)


In [ ]:
#@title 💡 Solution — Problem 23
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.T.flatten()


example = t.arange(6).reshape(2, 3)
print(solve(example))


<!-- dd:dd-q492 -->

### Problem 492 · independent

Write a function solve(n, rows) that returns a 2-D tensor holding the integers 0 through n-1 in order, arranged into `rows` rows. n is always divisible by rows. Build the run of values first, then re-describe its shape.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [3, 4, 5]])
```


In [ ]:
import torch as t

def solve(n, rows):
    """Return 0..n-1 laid out with `rows` rows."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (6, 2)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(492)


In [ ]:
#@title 💡 Solution — Problem 492
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, rows):
    """Return 0..n-1 laid out with `rows` rows."""
    return t.arange(n).reshape(rows, -1)


example = (6, 2)
print(solve(*example))


<!-- dd:dd-q493 -->

### Problem 493 · independent

Write a function solve(x) that returns a tuple (values, shape). `values` is a plain Python list of x's elements in row-major order; `shape` is x's shape as a plain tuple of ints. Together these are everything you would need to rebuild x — which is the whole point of reshape being free: the values never move.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([1, 2, 3, 4, 5, 6], (2, 3))
```


In [ ]:
import torch as t

def solve(x):
    """Return (flattened copy, shape before flattening)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [4, 5, 6]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(493)


In [ ]:
#@title 💡 Solution — Problem 493
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (flattened copy, shape before flattening)."""
    return (x.flatten().tolist(), tuple(x.shape))


example = t.tensor([[1, 2, 3], [4, 5, 6]])
print(solve(example))


<!-- dd:dd-q494 -->

### Problem 494 · independent

Write a function solve(x) that takes a CONTIGUOUS 2-D integer tensor — one built directly, not a transpose or other rearranged view — reshapes it to 1-D, writes 99 into the first element of the RESULT, and returns a tuple (result_values, x_values_now) with both as plain lists. Reshaping a contiguous tensor hands back a view over the same memory, so watch what happens to x. The contiguity restriction is the point, not fine print: reshape can only give you a view when the values are already laid out in the order the new shape wants, and it silently copies when they are not.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([99, 2, 3, 4], [99, 2, 3, 4])
```


In [ ]:
import torch as t

def solve(x):
    """Return a reshaped VIEW and prove it shares storage."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2], [3, 4]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(494)


In [ ]:
#@title 💡 Solution — Problem 494
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return a reshaped VIEW and prove it shares storage."""
    v = x.reshape(-1)
    v[0] = 99
    return (v.tolist(), x.flatten().tolist())


example = t.tensor([[1, 2], [3, 4]])
print(solve(example))


#### Common mistakes

- **"Reshape can rearrange values."** — Reshape never *reorders* values: the
  flat row-major sequence of elements is identical before and after, and only
  the shape metadata changes. (It may still copy that sequence into fresh
  memory when the input is a non-contiguous view — same order, new buffer.)
  If the values need to move (transpose, sort, flip), reshape is the wrong
  tool.
- **"reshape(3, 4) on 11 elements will pad or truncate."** — It raises a
  `RuntimeError`. Counts must match exactly; `-1` only *derives* a dimension,
  it can't invent elements.
- **"Flattening reads down the columns."** — The order is row-major: across
  row 0 first. Column-major means transposing first.
- **"`order='F'` works, like in NumPy."** — PyTorch's `flatten` takes no order
  argument at all. `z.T.flatten()` is the column-major read.


<!-- dd:dd-kp-numpy-elementwise-ufuncs -->

## Elementwise math

`numpy.elementwise-ufuncs`


### write the formula once — operators are elementwise


The core promise of tensor programming: **write the formula once, and it
applies to every element** — no loop. Operators `+ - * / ** %` between a
tensor and a scalar, or between two same-shaped tensors, work element by
element: `z * 2` doubles everything; `a * b` multiplies corresponding entries
(NOT matrix multiplication — that's `@`).

The general procedure for any "transform each entry" task:

> Express the rule for ONE element as a formula, then write that formula
> with the whole tensor in place of the element.


In [ ]:
import torch as t

z = t.tensor([1.0, 2.0, 3.0, 4.0])
print(z * 2)
print(z ** 2 - 1)
print(z % 2)




"Replace each x by x² − 1" → `z**2 - 1`. If you find yourself writing
`for i in range(len(z))`, stop — the elementwise spelling is shorter and
orders of magnitude faster (the loop happens in compiled code, and on a GPU it
happens in parallel). All of these return **new tensors** and leave the input
untouched.


In [ ]:
before = z.tolist()
squared = z ** 2 - 1
print("z after the expression:", z)
assert z.tolist() == before          # nothing was written back




And the one operator that is NOT elementwise, so the contrast lands early:


In [ ]:
a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print("a * a (elementwise):")
print(a * a)
print("a @ a (matrix product):")
print(a @ a)
assert not t.equal(a * a, a @ a)


In [ ]:
import torch as t

z = t.tensor([1.0, 2.0, 3.0])

# The per-element rule "x**2 - 1", written once for the whole tensor:
out = z**2 - 1
assert out.tolist() == [0.0, 3.0, 8.0]

# The input is untouched — the expression built a new tensor.
assert z.tolist() == [1.0, 2.0, 3.0]
print("z  ", z)
print("out", out)




Why: the expression reads exactly like the per-element rule — that
transliteration IS the method.


<!-- dd:dd-q192 -->

### Problem 192 · faded — your turn

Elementwise cube.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([  1.,   8., -27.])
```


In [ ]:
import torch as t

def solve(x):
    """Each entry raised to the third power."""
    return x _____ 3


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(192)


In [ ]:
#@title 💡 Solution — Problem 192
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.pow(x, 3)


print(solve(t.tensor([1.0, 2.0, -3.0])))


### named math functions and the rounding family


Beyond operators, named functions map over the whole tensor: `t.sqrt`,
`t.abs`, `t.exp`, `t.log`, `t.sin`, …

Rounding is a *family*, and the members differ on negatives:

- `t.round` — nearest;
- `t.floor` — largest integer ≤ x, so −0.3 → −1.0 (away from zero);
- `t.ceil` — smallest integer ≥ x;
- `t.trunc` — toward zero, so −0.3 → −0.0 (same as `.to(t.int64)`).

Read the task's example values to see which member is being asked for — and
the fastest way to tell them apart is to run all four on the same negatives:


In [ ]:
import torch as t

v = t.tensor([1.7, -0.3, 2.5, -2.5])
print("input", v)
print("round", t.round(v))
print("floor", t.floor(v))
print("ceil ", t.ceil(v))
print("trunc", t.trunc(v))




The only column where floor and trunc disagree is the negative one, which is
exactly where a grader will look:


In [ ]:
assert t.floor(v).tolist() == [1.0, -1.0, 2.0, -3.0]
assert t.trunc(v).tolist() == [1.0, -0.0, 2.0, -2.0]
assert t.equal(t.trunc(v).to(t.int64), v.to(t.int64))
print("at -0.3: floor ->", t.floor(v)[1].item(),
      "but trunc ->", t.trunc(v)[1].item())
print("trunc == .to(t.int64):", bool(t.equal(t.trunc(v).to(t.int64),
                                             v.to(t.int64))))


In [ ]:
import torch as t

v = t.tensor([1.7, -0.3, 2.5])

assert t.floor(v).tolist() == [1.0, -1.0, 2.0]   # floor moves DOWN
assert t.trunc(v).tolist() == [1.0, -0.0, 2.0]   # trunc moves toward zero
assert t.sqrt(t.tensor([4.0, 9.0])).tolist() == [2.0, 3.0]
print("input", v)
print("floor", t.floor(v))
print("trunc", t.trunc(v), "  <- differs only at the negative")




Why: floor vs trunc only disagree on negatives — that's exactly where tasks
(and graders) check.


<!-- dd:dd-q49 -->

### Problem 49 · faded — your turn

Floor every entry (note what floor does to negatives).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 1.,  3., -1.])
```


In [ ]:
import torch as t

def solve(z):
    """Replace each entry by the largest integer value <= it."""
    return t._____(z)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(49)


In [ ]:
#@title 💡 Solution — Problem 49
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.floor(z)


example = t.tensor([1.2, 3.7, -0.3])
print(solve(example))


### elementwise choosers — maximum, minimum, clamp


`t.maximum(a, b)` / `t.minimum(a, b)` pick the larger/smaller *at each
position* (contrast with `a.max()`, which reduces the whole tensor to one
number — different KP). `z.clamp(min=lo, max=hi)` limits values to a range;
NumPy calls this `clip`, and PyTorch accepts that spelling as an alias, but
`clamp` is the name you will read in model code.


In [ ]:
import torch as t

a = t.tensor([1.0, 5.0, 2.0])
b = t.tensor([3.0, 4.0, 2.5])
print("maximum:", t.maximum(a, b))     # one answer per position
print("a.max():", a.max())             # one answer, full stop




These overlap: `clamp(max=100)` and `t.minimum(x, ...)` compute the same
thing, and `clamp(min=0)` is ReLU. Prefer `clamp` for a constant bound — it
takes a plain Python number, whereas `t.minimum` wants a second tensor, and a
tensor you construct on the spot lands on the CPU and will not match an input
living on a GPU.


In [ ]:
readings = t.tensor([-2.0, 0.5, 7.0, 12.0])
print("clamp(min=0):     ", readings.clamp(min=0.0))       # ReLU
print("clamp(max=10):    ", readings.clamp(max=10.0))
print("clamp(0, 10):     ", readings.clamp(min=0.0, max=10.0))
assert t.equal(readings.clamp(max=10.0), t.minimum(readings, t.tensor(10.0)))


In [ ]:
import torch as t

# Elementwise chooser between TWO tensors: keep the larger at each slot.
a = t.tensor([1.0, 5.0, 2.0])
b = t.tensor([3.0, 4.0, 2.5])
assert t.maximum(a, b).tolist() == [3.0, 5.0, 2.5]

# Pipeline: curve exam scores — add 5, cap at 100, floor to whole points.
scores = t.tensor([71.5, 88.25, 97.0, 99.5])
curved = t.floor((scores + 5).clamp(max=100.0))
assert curved.tolist() == [76.0, 93.0, 100.0, 100.0]
print("raw   ", scores)
print("curved", curved)

# The other bound. `min=` sets a FLOOR, `max=` sets a CEILING — and the two
# read backwards from how they sound: min= raises everything below it.
readings = t.tensor([-2.0, 0.5, 7.0])
assert readings.clamp(min=0.0).tolist() == [0.0, 0.5, 7.0]
print("readings   ", readings)
print("clamp(min=0)", readings.clamp(min=0.0), " <- this is ReLU")




Why: composing these left-to-right is normal style — each stage maps over
the whole tensor, and the pipeline reads exactly like the per-element rule:
add, cap, floor.

The last line is the one worth memorising: `clamp(min=0.0)` is ReLU, the
single most common nonlinearity in the whole field.


<!-- dd:dd-q67 -->

### Problem 67 · faded — your turn

Negatives become 0.0, non-negatives pass through (ReLU).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.0000, 0.5000, 3.0000])
```


In [ ]:
import torch as t

def solve(z):
    """Each negative entry replaced by 0.0 (new tensor; z unmodified)."""
    return z.clamp(_____=0.0)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(67)


In [ ]:
#@title 💡 Solution — Problem 67
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.clamp(min=0.0)


example = t.tensor([-2.0, 0.5, 3.0])
print(solve(example))


<!-- dd:dd-q487 -->

### Problem 487 · guided

Write a function solve(x) that takes a PyTorch tensor and returns a new tensor with every element doubled. Write the formula once for the whole tensor — there is no loop to write.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 2., -4.,  7.])
```


<details>
<summary>Hints</summary>

1. The whole tensor at once — there is no index to loop over.
2. An arithmetic operator applied to a tensor is applied to every element.
3. `x * 2`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return x with every element doubled."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([1.0, -2.0, 3.5])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(487)


In [ ]:
#@title 💡 Solution — Problem 487
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return x with every element doubled."""
    return x * 2


example = t.tensor([1.0, -2.0, 3.5])
print(solve(example))


<!-- dd:dd-q488 -->

### Problem 488 · guided

Write a function solve(x) that takes a PyTorch tensor of floats and returns a new tensor holding the absolute value of each element. The original must be left unchanged.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1.5000, 2.0000, 3.2500])
```


<details>
<summary>Hints</summary>

1. You need the magnitude of each element, sign discarded.
2. It is a method on the tensor, and it returns a NEW tensor rather than
   editing yours — which is what keeps the input unchanged.
3. `x.abs()`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return the absolute value of every element."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([-1.5, 2.0, -3.25])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(488)


In [ ]:
#@title 💡 Solution — Problem 488
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return the absolute value of every element."""
    return x.abs()


example = t.tensor([-1.5, 2.0, -3.25])
print(solve(example))


<!-- dd:dd-q43 -->

### Problem 43 · independent

Write a function solve(a, b) that takes two 1-D PyTorch tensors of equal length and returns a new tensor of the same length, where each element is the larger of the two corresponding elements from a and b. If the two values at a position are equal, keep that shared value. Do not use Python loops or Python's built-in max.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([4., 5., 3., 7.])
```


In [ ]:
import torch as t

def solve(a, b):
    """Return the elementwise maximum of a and b."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
a = t.tensor([1.0, 5.0, 3.0, 2.0])
b = t.tensor([4.0, 2.0, 3.0, 7.0])
print(solve(a, b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(43)


In [ ]:
#@title 💡 Solution — Problem 43
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.maximum(a, b)


a = t.tensor([1.0, 5.0, 3.0, 2.0])
b = t.tensor([4.0, 2.0, 3.0, 7.0])
print(solve(a, b))


<!-- dd:dd-q489 -->

### Problem 489 · independent

Write a function solve(x, lo, hi) that takes a float tensor and two floats, and returns a new tensor in which every element below lo has been raised to lo and every element above hi lowered to hi. Values already inside the range are left alone. Remember that min= sets the FLOOR and max= the CEILING — they read backwards from how they sound.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.0000, 0.5000, 5.0000])
```


In [ ]:
import torch as t

def solve(x, lo, hi):
    """Return x with every element pulled into [lo, hi]."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([-3.0, 0.5, 9.0]), 0.0, 5.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(489)


In [ ]:
#@title 💡 Solution — Problem 489
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, lo, hi):
    """Return x with every element pulled into [lo, hi]."""
    return x.clamp(min=lo, max=hi)


example = (t.tensor([-3.0, 0.5, 9.0]), 0.0, 5.0)
print(solve(*example))


<!-- dd:dd-q63 -->

### Problem 63 · independent

Write a function solve(z) that takes a 1-D PyTorch tensor of POSITIVE floats and returns a float tensor of the same shape holding just the integer part of each entry (e.g. 3.7 becomes 3.0). Several torch idioms compute this — use whichever you like.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([3., 0., 5.])
```


In [ ]:
import torch as t

def solve(z):
    """Return the integer part of each positive entry of z, as floats."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([3.7, 0.2, 5.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(63)


In [ ]:
#@title 💡 Solution — Problem 63
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.floor(z)


example = t.tensor([3.7, 0.2, 5.0])
print(solve(example))


#### Common mistakes

- **"`a * b` on two matrices is matrix multiplication."** — It is elementwise.
  Matrix product is `a @ b`. This distinction matters enough that it gets its
  own KP later.
- **"`t.maximum` and `t.max` are the same."** — `t.maximum(a, b)` compares
  two tensors position-by-position (returns a tensor); `t.max(a)` reduces one
  tensor to its single largest value.
- **"floor and truncate are the same."** — Only for positives. For negatives,
  floor moves AWAY from zero (−0.3 → −1.0) while trunc moves toward it
  (−0.3 → −0.0). Read the task's example values to see which is being asked.
- **"An in-place version is just a style choice."** — Methods ending in an
  underscore (`z.clamp_(min=0)`) modify the tensor you were handed. That is a
  different contract from `z.clamp(min=0)`, and it is how you accidentally
  mutate a caller's data.


<!-- dd:dd-kp-numpy-aggregations -->

## Whole-tensor aggregations and Python scalars

`numpy.aggregations`


### reductions — collapsing a tensor to one number


Where an elementwise operation maps a tensor to a same-shaped tensor, an
**aggregation (reduction)** collapses a tensor down to a single number:
`x.sum()`, `x.mean()`, `x.min()`, `x.max()`, `x.std()` — the workhorses.

Called with no arguments, each of these reduces over **all** elements
regardless of shape — a 2-D tensor's `x.max()` is the max of the whole matrix.
(Reducing along just one axis is the `dim=` keyword, which gets its own KP in
the broadcasting lesson — walk before running.)


In [ ]:
import torch as t

grid = t.tensor([[3.0, 8.0, 1.0],
                 [6.0, 2.0, 9.0]])
print("sum ", grid.sum())
print("mean", grid.mean())
print("min ", grid.min())
print("max ", grid.max())




Six values in, one value out, every time — and notice the shape never came
into it:


In [ ]:
flat = grid.reshape(6)
assert t.equal(flat.max(), grid.max())
assert t.equal(flat.sum(), grid.sum())
print(grid.shape, "and", flat.shape, "→ same answers")


Task: global min and max of a matrix of sensor readings.


In [ ]:
import torch as t

readings = t.tensor([[3.5, -2.0, 7.25],
                     [0.0,  9.5, -8.75]])

# min/max with no dim argument scan the WHOLE tensor, ignoring shape.
lo, hi = readings.min(), readings.max()
assert (lo.item(), hi.item()) == (-8.75, 9.5)
print("min", lo.item(), "| max", hi.item())




Why: no dim argument = one value for the whole tensor, shape ignored. That's
the default to internalize before `dim=` complicates things.


<!-- dd:dd-q26 -->

### Problem 26 · faded — your turn

Global min and max of a 2-D tensor, returned as a (min, max) pair of plain
Python numbers.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(-8.75, 9.5)
```


In [ ]:
import torch as t

def solve(x):
    """Return (smallest, largest) element of the whole 2-D tensor."""
    return (x._____().item(), x._____().item())


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(26)


In [ ]:
#@title 💡 Solution — Problem 26
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return (x.min().item(), x.max().item())


example = t.tensor([[3.5, -2.0, 7.25], [0.0, 9.5, -8.75]])
print(solve(example))


### 0-dimensional tensors vs plain Python numbers


One practical wrinkle, and it bites harder here than in NumPy: reductions
return a **0-dimensional tensor**, not a number. It prints as
`tensor(1.5833)` and it still carries a dtype, a device, and possibly a
gradient. Graders, JSON encoders, and f-strings care.

When a task says "return a plain Python int/float/bool", convert explicitly:

> `float(x.mean())`, `int(x.sum())`, `bool((x > 0).any())`
> — or `x.item()`, the generic "unwrap this 0-d result".


In [ ]:
import torch as t

grid = t.tensor([[3.0, 8.0, 1.0],
                 [6.0, 2.0, 9.0]])
raw = grid.mean()
print(raw, "| ndim", raw.ndim, "| type", type(raw).__name__)
print(float(raw), "| type", type(float(raw)).__name__)




The first line still says `tensor(...)`. That is the whole distinction:


In [ ]:
assert raw.ndim == 0 and isinstance(raw, t.Tensor)
assert isinstance(raw.item(), float)
assert int(grid.sum()) == 29
assert bool((grid > 0).all()) is True
print("raw.ndim", raw.ndim, "-> still a tensor:", isinstance(raw, t.Tensor))
print("int(grid.sum())", int(grid.sum()), type(int(grid.sum())).__name__)
print("bool((grid > 0).all())", bool((grid > 0).all()),
      type(bool((grid > 0).all())).__name__)




Keep tensors *inside* your computation; convert exactly at the boundary
where a plain Python value is required. Unwrapping early is how you
accidentally break the autograd chain in real model code.


In [ ]:
import torch as t

readings = t.tensor([[3.5, -2.0, 7.25],
                     [0.0,  9.5, -8.75]])

# mean returns a 0-d TENSOR. Usually fine, but when the contract says
# "a single float scalar", unwrap it explicitly.
raw = readings.mean()
assert raw.ndim == 0 and isinstance(raw, t.Tensor)

avg = float(raw)
assert isinstance(avg, float)
assert abs(avg - 1.5833333) < 1e-5
print("as a tensor:", raw, "| as a float:", avg)




Why: `float(...)` at the boundary — the computation stays in torch, only the
returned value is unwrapped.


<!-- dd:dd-q28 -->

### Problem 28 · faded — your turn

The arithmetic mean of a vector, as a plain Python float.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
2.5
```


In [ ]:
import torch as t

def solve(x):
    """Mean of x as a plain Python float."""
    return _____(x.mean())


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(28)


In [ ]:
#@title 💡 Solution — Problem 28
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return float(x.mean())


example = t.tensor([1.0, 2.0, 3.0, 4.0])
print(solve(example))


### whole-tensor yes/no verdicts


Yes/no questions about tensors have two standard shapes:

- **Comparison, then reduce.** A comparison builds a boolean tensor (previous
  KP); `x.any()` (is at least one entry True?) or `x.all()` (are they all
  True?) collapses it to one answer. `(x > 0).all()` asks "is everything
  positive?".
- **Whole-tensor equality.** `t.equal(a, b)` is an exact match of shape
  and values; `t.allclose(a, b)` is equality within floating-point tolerance
  — the right check after float arithmetic.


In [ ]:
import torch as t

grid = t.tensor([[3.0, -8.0, 1.0],
                 [6.0, 2.0, -9.0]])
print("negatives present?", bool((grid < 0).any()))
print("all positive?    ", bool((grid > 0).all()))




`a == b` alone is NOT a verdict — it's elementwise and yields a boolean
tensor (and `if` on it raises an error).


In [ ]:
a = t.tensor([1.0, 2.0])
b = t.tensor([1.0, 5.0])
print(a == b)                 # a tensor, not an answer
print(t.equal(a, b))          # one bool




And the reason `allclose` exists at all — float arithmetic does not land
where the arithmetic says it should:


In [ ]:
summed = t.full((10,), 0.1).sum()
print(summed.item(), "vs", 1.0)
assert not t.equal(summed, t.tensor(1.0))
assert t.allclose(summed, t.tensor(1.0))


In [ ]:
import torch as t

readings = t.tensor([[3.5, -2.0, 7.25],
                     [0.0,  9.5, -8.75]])

# Boolean pipeline: comparison (elementwise) then reduction (any).
# Read it aloud: "readings less than zero — any?"
has_negative = bool((readings < 0).any())
assert has_negative is True

# Float-safe equality: after arithmetic, prefer allclose. Ten 0.1s summed
# in float32 land just past 1.0.
a = t.full((10,), 0.1).sum()
b = t.tensor(1.0)
assert not t.equal(a, b)         # bitwise-exact? no — accumulated float error
assert t.allclose(a, b)          # equal within tolerance? yes
print("any negative?", has_negative)
print("ten 0.1s summed:", a.item(), "| equal:", bool(t.equal(a, b)),
      "| allclose:", bool(t.allclose(a, b)))




Why: exact equality is for ints/bools and provenance checks; `allclose` is
for anything that went through float arithmetic — and float32 has fewer
digits to spare than the float64 you may be used to.


<!-- dd:dd-q64 -->

### Problem 64 · faded — your turn

Tolerant closeness AND exact equality of two tensors, as two plain bools.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, True)
```


In [ ]:
import torch as t

def solve(a, b):
    """(close within float tolerance?, exactly equal?) as plain bools."""
    return (bool(t._____(a, b)), bool(t._____(a, b)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(64)


In [ ]:
#@title 💡 Solution — Problem 64
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return bool(t.allclose(a, b)), bool(t.equal(a, b))


print(solve(t.tensor([1.0, 2.0]), t.tensor([1.0, 2.0])))


<!-- dd:dd-q495 -->

### Problem 495 · guided

Write a function solve(x) that takes a PyTorch tensor and returns the sum of ALL its elements as a plain Python number, not a tensor. A whole-tensor reduction gives you a 0-dimensional tensor; .item() is how you get the number out of it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
10.0
```


<details>
<summary>Hints</summary>

1. Two steps: collapse the tensor to one number, then leave PyTorch behind.
2. A whole-tensor reduction returns a 0-dimensional TENSOR, not a Python
   number — the test checks which one you handed back.
3. `x.sum().item()`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return the sum of every element as a Python number."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(495)


In [ ]:
#@title 💡 Solution — Problem 495
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return the sum of every element as a Python number."""
    return x.sum().item()


example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


<!-- dd:dd-q496 -->

### Problem 496 · guided

Write a function solve(x) that returns a tuple (mean, count): the mean of every element as a plain Python float, and the total number of elements as a plain int. Both are whole-tensor questions — no axis is involved.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2.5, 4)
```


<details>
<summary>Hints</summary>

1. Both halves are whole-tensor questions, so neither needs an axis.
2. The count is metadata you already know how to read — it is not a
   reduction, and it does not need .item().
3. `(x.mean().item(), x.numel())`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return (mean, count) over the whole tensor."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(496)


In [ ]:
#@title 💡 Solution — Problem 496
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (mean, count) over the whole tensor."""
    return (x.mean().item(), x.numel())


example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


<!-- dd:dd-q62 -->

### Problem 62 · independent

Write a function solve(z) that takes a 1-D PyTorch integer tensor and returns the sum of all its entries as a plain Python int — not a 0-dimensional tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
45
```


In [ ]:
import torch as t

def solve(z):
    """Return the sum of z's entries as a Python int."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.arange(10)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(62)


In [ ]:
#@title 💡 Solution — Problem 62
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return int(z.sum())


example = t.arange(10)
print(solve(example))


<!-- dd:dd-q497 -->

### Problem 497 · independent

Write a function solve(x, threshold) that returns a tuple (any_above, how_many): a plain Python bool saying whether ANY element exceeds the threshold, and a plain int counting how many do. Both come from the same boolean tensor — build it once. True counts as 1 when you sum it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(True, 2)
```


In [ ]:
import torch as t

def solve(x, threshold):
    """Return (any above threshold?, how many)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.0, 5.0, 9.0]), 4.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(497)


In [ ]:
#@title 💡 Solution — Problem 497
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, threshold):
    """Return (any above threshold?, how many)."""
    mask = x > threshold
    return (bool(mask.any()), int(mask.sum()))


example = (t.tensor([1.0, 5.0, 9.0]), 4.0)
print(solve(*example))


<!-- dd:dd-q498 -->

### Problem 498 · independent

Write a function solve(x) that returns a tuple (min, max, span) of plain Python floats: the smallest element, the largest, and the distance between them. Reduce twice and do the subtraction in Python — the span is not itself a reduction.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(-2.0, 7.25, 9.25)
```


In [ ]:
import torch as t

def solve(x):
    """Return (min, max, span) as plain Python floats."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[3.5, -2.0], [7.25, 0.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(498)


In [ ]:
#@title 💡 Solution — Problem 498
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return (min, max, span) as plain Python floats."""
    lo = x.min().item()
    hi = x.max().item()
    return (lo, hi, hi - lo)


example = t.tensor([[3.5, -2.0], [7.25, 0.0]])
print(solve(example))


#### Common mistakes

- **"`x.max()` on a matrix gives per-row maxima."** — With no arguments it
  reduces over everything: one value for the whole tensor. Per-row/column
  reductions need `dim=`, covered in the broadcasting lesson.
- **"Reductions return normal Python numbers."** — They return 0-dimensional
  tensors. Mostly interchangeable in arithmetic, but "return a plain
  int/float" contracts require `int(...)`/`float(...)`/`.item()`.
- **"`==` tells me whether two tensors are equal."** — `a == b` is ELEMENTWISE,
  yielding a boolean tensor (and `if` on it raises an error). Whole-tensor
  verdicts are `t.equal` (exact) or `t.allclose` (float-tolerant).
- **"`t.array_equal` is the exact check."** — That's the NumPy name; there is
  no such function here. It's `t.equal`.


<!-- dd:dd-kp-numpy-sorting -->

## Sorting tensors

`numpy.sorting`


### sort returns a pair, not a tensor


**`t.sort(z)` does not return a sorted tensor.** It returns a *pair* — the
sorted values and the indices that produced them — and forgetting that is the
mistake this KP exists to prevent:

```python no-run
t.sort(z)          # -> torch.return_types.sort(values=..., indices=...)
t.sort(z).values   # the sorted tensor you actually wanted
```


In [ ]:
import torch as t

z = t.tensor([0.5, 0.25, 0.75])
print(t.sort(z))




That printout is the pair, and it is what a function returns if you forget
`.values`.

You can unpack it either way: `values, indices = t.sort(z)`, or reach for
`.values` / `.indices` by name. The indices half is not a consolation prize —
it is what "sort one thing by another" tasks need, and it is the same thing
`t.argsort(z)` gives you on its own.


In [ ]:
values, indices = t.sort(z)
print("values ", values)
print("indices", indices)
assert t.equal(indices, t.argsort(z))
assert t.equal(z[indices], values)      # the indices reconstruct the values




Sorting never modifies the input: `t.sort(z)` and the method form `z.sort()`
both leave `z` alone and hand back a new pair. (There is no in-place `sort_`.)

**`descending=True`** sorts largest-first — a real keyword, unlike NumPy, where
you have to sort ascending and reverse afterwards.


In [ ]:
print("z is still", z)
print("descending", t.sort(z, descending=True).values)
assert z.tolist() == [0.5, 0.25, 0.75]


Task: produce a sorted copy of a vector, confirm the original is intact, then
get the same values descending.


In [ ]:
import torch as t

z = t.tensor([0.5, 0.25, 0.75])

# sort returns a PAIR — take .values for the sorted tensor.
result = t.sort(z)
assert result.values.tolist() == [0.25, 0.5, 0.75]
assert result.indices.tolist() == [1, 0, 2]     # where each value came from

# The input keeps its original order (the grader often checks this).
assert z.tolist() == [0.5, 0.25, 0.75]

# Descending is a keyword here — no reverse step needed.
desc = t.sort(z, descending=True).values
assert desc.tolist() == [0.75, 0.5, 0.25]

# Unpacking works too, and reads well when you want both halves.
values, indices = t.sort(z)
assert values.tolist() == [0.25, 0.5, 0.75]
print("input     ", z)
print("values    ", values)
print("indices   ", indices)
print("descending", desc)




Why each step:

1. Taking `.values` is the habit to build. Returning `t.sort(z)` straight from
   a function hands the caller a pair, and the failure looks like a type error
   far from the line that caused it.
2. The indices are the bridge to the order-statistics KP: they say *where*
   each sorted value came from, which is how you carry a second tensor along.
3. `descending=True` is one of the places PyTorch is friendlier than NumPy —
   worth knowing so you don't write the reverse-slice workaround (which
   wouldn't work here anyway, since negative slice steps are rejected).


<!-- dd:dd-q58 -->

### Problem 58 · faded — your turn

Sorted copy, smallest to largest, input left unmodified.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.1000, 0.4000, 0.9000])
```


In [ ]:
import torch as t

def solve(z):
    """Return a NEW tensor with z's values in ascending order."""
    return t.sort(z)._____


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(58)


In [ ]:
#@title 💡 Solution — Problem 58
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.sort(z).values


example = t.tensor([0.4, 0.1, 0.9])
print(solve(example))


### argsort — the positions, and reordering by them


`t.argsort(z)` gives you the indices half on its own: `order[0]` is the position
of the smallest element, `order[1]` the next, and so on. On a single tensor that
is just a slower route to `t.sort(z).values` — you would still have to index with
it.

Its real job is **carrying a second tensor along**. When two tensors are parallel
— names and scores, boxes and confidences, tokens and logits — sorting one of
them independently destroys the correspondence. So you rank once, get the order,
and index *every* tensor with that same order. They stay lined up because they
all moved the same way.

Indexing a tensor with an index tensor is the "fancy indexing" you have already
met: `names[order]` builds a new tensor by reading `names` at each position in
`order`, in that order.


In [ ]:
import torch as t

ids = t.tensor([10, 11, 12, 13])
scores = t.tensor([0.4, 0.9, 0.1, 0.7])

order = t.argsort(scores, descending=True)
print("order  ", order)
print("ids    ", ids[order])
print("scores ", scores[order])




Both tensors moved by the SAME order, so row-by-row they still describe the
same items. Sorting them separately is the bug this pattern prevents:


In [ ]:
broken = t.sort(ids, descending=True).values
print("independently sorted ids:", broken, "— no longer paired with anything")
assert t.equal(scores[order], t.sort(scores, descending=True).values)


In [ ]:
import torch as t

names = t.tensor([10, 20, 30])
scores = t.tensor([0.5, 0.25, 0.75])

# The positions that WOULD sort scores ascending — not the values.
order = t.argsort(scores)
assert order.tolist() == [1, 0, 2]

# Index both tensors with the SAME order and they stay in correspondence.
assert scores[order].tolist() == [0.25, 0.5, 0.75]
assert names[order].tolist() == [20, 10, 30]

# argsort takes the same direction keyword sort does.
best_first = t.argsort(scores, descending=True)
assert best_first.tolist() == [2, 0, 1]
assert names[best_first].tolist() == [30, 10, 20]
print("scores       ", scores, " names", names)
print("best-first   ", best_first)
print("names ranked ", names[best_first], " scores", scores[best_first])




Read the last two lines: name 30 comes first because score 0.75 is the highest,
and nothing about `names` was sorted. That is the whole pattern — rank one
tensor, index all of them.


<!-- dd:dd-q520 -->

### Problem 520 · faded — your turn

Rank by score, then move both tensors with the same order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([20, 10, 30], [0.8999999761581421, 0.4000000059604645, 0.10000000149011612])
```


In [ ]:
import torch as t

def solve(names, scores):
    """Return (names, scores) as lists, both ordered highest score first."""
    order = t.argsort(scores, _____=True)
    return (names[_____].tolist(), scores[_____].tolist())


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(520)


In [ ]:
#@title 💡 Solution — Problem 520
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(names, scores):
    """Reorder one tensor by another's ranking."""
    order = t.argsort(scores, descending=True)
    return (names[order].tolist(), scores[order].tolist())


example = (t.tensor([10, 20, 30]), t.tensor([0.4, 0.9, 0.1]))
print(solve(*example))


### picking an axis, and taking only the top k


On a tensor with more than one axis, **`dim=`** says which axis to sort along,
and every lane along that axis is sorted independently. `t.sort(m, dim=1)` sorts
*within* each row — which means it scrambles each row's contents and destroys any
correspondence between columns. That is usually not what you want from "sort the
matrix"; reordering whole rows is the argsort-plus-indexing pattern from the last
segment, applied to axis 0.


In [ ]:
import torch as t

m = t.tensor([[3.0, 1.0, 2.0],
              [9.0, 7.0, 8.0]])
print("dim=1 (within each row)")
print(t.sort(m, dim=1).values)
print("dim=0 (down each column)")
print(t.sort(m, dim=0).values)




**`t.topk(z, k)`** answers a narrower question: the k largest values, already
ordered largest first, plus their positions. Sorting the whole tensor to slice
off k of them does strictly more work — and on a long vector, a great deal more.
Like `sort`, it hands back a `(values, indices)` pair.


In [ ]:
v = t.tensor([5.0, 1.0, 9.0, 3.0, 7.0])
top = t.topk(v, 2)
print(top)
assert top.values.tolist() == [9.0, 7.0]
assert t.equal(top.values, t.sort(v, descending=True).values[:2])


In [ ]:
import torch as t

m = t.tensor([[3.0, 1.0, 2.0], [9.0, 7.0, 8.0]])

# dim=1 sorts WITHIN each row, independently of the other rows.
assert t.sort(m, dim=1).values.tolist() == [[1.0, 2.0, 3.0], [7.0, 8.0, 9.0]]

# dim=0 does the same down the columns.
assert t.sort(m, dim=0).values.tolist() == [[3.0, 1.0, 2.0], [9.0, 7.0, 8.0]]

# argsort takes the same dim= keyword, and answers with positions not values.
assert t.argsort(m, dim=1).tolist() == [[1, 2, 0], [1, 2, 0]]

# ...and the same descending= keyword you met a segment ago.
z = t.tensor([0.5, 0.25, 0.75, 0.125])
assert t.argsort(z, descending=True).tolist() == [2, 0, 1, 3]

# topk: the k largest, largest first — a pair again, so take .values.
top = t.topk(z, 2)
assert top.values.tolist() == [0.75, 0.5]
assert top.indices.tolist() == [2, 0]
print("dim=1 (within rows)")
print(t.sort(m, dim=1).values)
print("dim=0 (down columns) — unchanged, columns were already ascending")
print(t.sort(m, dim=0).values)
print("topk:", top)




The second assertion is the one to sit with: sorting down the columns left this
matrix unchanged, because every column was already ascending. Sorting along an
axis tells you nothing about the other axis.


<!-- dd:dd-q521 -->

### Problem 521 · faded — your turn

Per-row rankings, largest first — an axis and a direction at the same time.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 0, 2],
        [0, 2, 1]])
```


In [ ]:
import torch as t

def solve(x):
    """Return each row's column indices ordered by that row's values, largest first."""
    return t.argsort(x, _____=1, _____=True)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(521)


In [ ]:
#@title 💡 Solution — Problem 521
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return each row's ranking positions, descending."""
    return t.argsort(x, dim=1, descending=True)


example = t.tensor([[0.4, 0.9, 0.1], [5.0, 1.0, 3.0]])
print(solve(example))


<!-- dd:dd-q516 -->

### Problem 516 · guided

Write a function solve(z) that takes a 1-D float tensor and returns a NEW tensor with the same values ordered from largest to smallest. The input must be left unchanged.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2.0000, 1.0000, 0.5000])
```


<details>
<summary>Hints</summary>

1. A NEW tensor, so the input has to survive untouched.
2. Sorting returns both values and the positions they came from; you want
   the values half, and there is a keyword for the direction.
3. `t.sort(z, descending=True).values`.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return the values of z, largest first."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([0.5, 2.0, 1.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(516)


In [ ]:
#@title 💡 Solution — Problem 516
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    """Return the values of z, largest first."""
    return t.sort(z, descending=True).values


example = t.tensor([0.5, 2.0, 1.0])
print(solve(example))


<!-- dd:dd-q517 -->

### Problem 517 · guided

Write a function solve(z) that takes a 1-D float tensor and returns a tensor of indices: the position each element would come from if z were sorted smallest to largest. Sorting gives you the values; argsort gives you where they came from — which is what you need when a second tensor has to be reordered the same way.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 0, 2])
```


<details>
<summary>Hints</summary>

1. Not the sorted values — where each sorted value WOULD have come from.
2. The result is an index tensor the same length as z, and result[0] is the
   position of the smallest element.
3. `t.argsort(z)`.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return the POSITIONS that would sort z ascending."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([0.4, 0.1, 0.9])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(517)


In [ ]:
#@title 💡 Solution — Problem 517
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    """Return the POSITIONS that would sort z ascending."""
    return t.argsort(z)


example = t.tensor([0.4, 0.1, 0.9])
print(solve(example))


<!-- dd:dd-q518 -->

### Problem 518 · independent

Write a function solve(x) that takes a 2-D float tensor and returns a tensor of the same shape in which each row has been sorted ascending on its own. Rows do not mix — name the axis you want sorted along.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 2., 3.],
        [7., 8., 9.]])
```


In [ ]:
import torch as t

def solve(x):
    """Sort each ROW of a 2-D tensor independently."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[3.0, 1.0, 2.0], [9.0, 7.0, 8.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(518)


In [ ]:
#@title 💡 Solution — Problem 518
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Sort each ROW of a 2-D tensor independently."""
    return t.sort(x, dim=1).values


example = t.tensor([[3.0, 1.0, 2.0], [9.0, 7.0, 8.0]])
print(solve(example))


<!-- dd:dd-q519 -->

### Problem 519 · independent

Write a function solve(z, k) that takes a 1-D float tensor and an int, and returns the k largest values ordered largest first. Sorting the whole tensor to take k of them does more work than the question asks for; there is an operation that answers exactly this.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.9000, 0.7000])
```


In [ ]:
import torch as t

def solve(z, k):
    """Return the k largest values, largest first."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([0.4, 0.9, 0.1, 0.7]), 2)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(519)


In [ ]:
#@title 💡 Solution — Problem 519
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, k):
    """Return the k largest values, largest first."""
    return t.topk(z, k).values


example = (t.tensor([0.4, 0.9, 0.1, 0.7]), 2)
print(solve(*example))


<!-- dd:dd-q522 -->

### Problem 522 · independent

Write a function solve(x, col) that returns the rows of the 2-D float tensor x reordered so that column `col` is ascending, as a plain nested list. Rows must stay intact — pull the key column out, rank it, and use that ranking to index the whole matrix along axis 0.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[1.0, 2.0], [2.0, 3.0], [3.0, 1.0]]
```


In [ ]:
import torch as t

def solve(x, col):
    """Sort the ROWS of a matrix by one column's values."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[3.0, 1.0], [1.0, 2.0], [2.0, 3.0]]), 0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(522)


In [ ]:
#@title 💡 Solution — Problem 522
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, col):
    """Sort the ROWS of a matrix by one column's values."""
    order = t.argsort(x[:, col])
    return x[order].tolist()


example = (t.tensor([[3.0, 1.0], [1.0, 2.0], [2.0, 3.0]]), 0)
print(solve(*example))


#### Common mistakes

- **"`t.sort(z)` returns the sorted tensor."** — It returns a
  `(values, indices)` pair. Take `.values`, or unpack both.
- **"`z.sort()` sorts in place, like the NumPy method."** — It does not. There
  is no in-place sort; the input is never modified.
- **"There's no descending option, so I'll reverse the result."** — There is:
  `descending=True`. And the NumPy reversal idiom `[::-1]` would raise here
  anyway.
- **"Sorting a 2-D tensor sorts the rows as units."** — `t.sort(z, dim=1)`
  sorts *within* each row independently, destroying row integrity. Keeping
  rows intact while reordering them is an argsort + indexing pattern (later
  KP).


<!-- dd:dd-kp-numpy-tile-repeat-meshgrid -->

## Tiling and repetition — repeat, repeat_interleave, meshgrid

`numpy.tile-repeat-meshgrid`


### Repeat each element with t.repeat_interleave


`t.repeat_interleave(x, k)` repeats each element of `x` before moving to the
next element.

```text
[1, 2, 3] → [1, 1, 1, 2, 2, 2, 3, 3, 3]
```


In [ ]:
import torch as t

x = t.tensor([1, 2, 3])
print(t.repeat_interleave(x, 3))




Use it when output groups copies of each individual value. `k` may also be a
tensor containing one repetition count per element.


In [ ]:
counts = t.tensor([1, 0, 4])
print(t.repeat_interleave(x, counts))
assert t.repeat_interleave(x, counts).tolist() == [1, 3, 3, 3, 3]




A count of 0 makes that value vanish — which is exactly how run-length
decoding handles an empty run.


> **Watch out.** Read expected output from left to right. If one value finishes all its copies
before the next value appears, use `repeat_interleave`.

This is the operation NumPy calls `repeat` — and PyTorch also has a `repeat`,
which does something else entirely. Getting the two names straight is the
whole point of this KP.


Task: repeat each element of `[4, 7, 9]` three consecutive times.


In [ ]:
import torch as t

x = t.tensor([4, 7, 9])
repeated = t.repeat_interleave(x, 3)

assert repeated.tolist() == [4, 4, 4, 7, 7, 7, 9, 9, 9]
print(repeated)




Why: `repeat_interleave` completes three copies of `4`, then three copies of
`7`, then three copies of `9`.


<!-- dd:dd-q35 -->

### Problem 35 · faded — your turn

Each element appears `k` times consecutively.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 1, 1, 2, 2, 2, 3, 3, 3])
```


In [ ]:
import torch as t

def solve(x, k):
    """Each element of x, repeated k times consecutively."""
    return t._____(x, k)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(35)


In [ ]:
#@title 💡 Solution — Problem 35
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, k):
    return t.repeat_interleave(x, k)


example = t.tensor([1, 2, 3])
print(solve(example, 3))


### Repeat a whole block with x.repeat


`x.repeat(k)` repeats `x` as one complete block.

```text
[1, 2, 3] → [1, 2, 3, 1, 2, 3]
```


In [ ]:
import torch as t

x = t.tensor([1, 2, 3])
print("repeat          ", x.repeat(2))
print("repeat_interleave", t.repeat_interleave(x, 2))




Same six numbers, different order — that is the entire distinction, and it is
worth seeing on one screen.

For a 2-D tensor, pass one repetition count per axis:
`block.repeat(row_repeats, column_repeats)`. `t.tile` is a NumPy-compatible
alias for the same behaviour, and it takes the counts as a tuple.


In [ ]:
block = t.tensor([[0, 1],
                  [1, 0]])
print(block.repeat(2, 3))
assert block.repeat(2, 3).shape == (4, 6)
assert t.equal(block.repeat(2, 3), t.tile(block, (2, 3)))


> **Watch out.** **`x.repeat` is NOT NumPy's `repeat`.** It is NumPy's `tile`: it lays whole
copies end to end. The elementwise one is `repeat_interleave`. If you
translate `np.repeat(x, 3)` to `x.repeat(3)` you get the right length and the
wrong order — a bug that survives any test that only checks shape.

The repetition counts follow axis order. First number repeats rows; second
number repeats columns.


Task: repeat one 2×2 checker block twice vertically and twice horizontally.


In [ ]:
import torch as t

block = t.tensor([[0, 1],
                  [1, 0]])
mosaic = block.repeat(2, 2)

assert mosaic.tolist() == [[0, 1, 0, 1],
                           [1, 0, 1, 0],
                           [0, 1, 0, 1],
                           [1, 0, 1, 0]]
print(mosaic)

# Same thing, NumPy-style spelling.
assert t.tile(block, (2, 2)).tolist() == mosaic.tolist()

# And the contrast that matters — same input, other operation:
assert t.repeat_interleave(t.tensor([1, 2, 3]), 2).tolist() == [1, 1, 2, 2, 3, 3]
assert t.tensor([1, 2, 3]).repeat(2).tolist() == [1, 2, 3, 1, 2, 3]
print(mosaic)




Why: `(2, 2)` lays out two copies along rows and two copies along columns.


<!-- dd:dd-q34 -->

### Problem 34 · faded — your turn

Repeat the entire sequence `k` times end to end.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 2., 3., 1., 2., 3.])
```


In [ ]:
import torch as t

def solve(x, k):
    """The whole of x laid end-to-end k times."""
    return x._____(k)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(34)


In [ ]:
#@title 💡 Solution — Problem 34
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, k):
    return x.repeat(k)


example = t.tensor([1.0, 2.0, 3.0])
print(solve(example, 2))


### Build coordinate grids with t.meshgrid


`t.meshgrid(x, y, indexing='xy')` turns two 1-D coordinate axes into two 2-D
matrices.

- `X` repeats the `x` coordinates across every row.
- `Y` repeats each `y` coordinate down its matching row.

Pairing `X[i, j]` with `Y[i, j]` gives one point from every possible
combination of `x` and `y`.


In [ ]:
import torch as t

x = t.tensor([1.0, 2.0, 3.0])
y = t.tensor([10.0, 20.0])
X, Y = t.meshgrid(x, y, indexing='xy')
print(X)
print(Y)
print("shapes:", X.shape, Y.shape)




Two matrices, not one tensor of pairs. Stack them and the points read out
directly — every combination, once each:


In [ ]:
print(t.stack([X, Y], dim=-1).reshape(-1, 2))
assert X.numel() == len(x) * len(y)




The `indexing` keyword is not decoration. Ask for `'ij'` and the same call
hands back the transpose:


In [ ]:
Xij, Yij = t.meshgrid(x, y, indexing='ij')
print("xy shape", X.shape, " ij shape", Xij.shape)
assert t.equal(Xij, X.T)


> **Watch out.** `meshgrid` returns one coordinate matrix per input axis—not one tensor of
coordinate pairs.

**State the `indexing` argument.** With `indexing='xy'` the outputs have shape
`(len(y), len(x))`, matching NumPy. With `indexing='ij'` they come out
transposed, shape `(len(x), len(y))`. PyTorch will not guess for you quietly —
omitting the argument warns and uses `'ij'`, so NumPy code ported without it
silently transposes.


Task: build coordinate matrices for three x-values and two y-values.


In [ ]:
import torch as t

x = t.tensor([1.0, 2.0, 3.0])
y = t.tensor([10.0, 20.0])
X, Y = t.meshgrid(x, y, indexing='xy')

assert X.tolist() == [[1.0, 2.0, 3.0],
                      [1.0, 2.0, 3.0]]
assert Y.tolist() == [[10.0, 10.0, 10.0],
                      [20.0, 20.0, 20.0]]
print(X)
print(Y)




Why: each column chooses an x-coordinate; each row chooses a y-coordinate.
Their matching positions enumerate the full grid.


<!-- dd:dd-q29 -->

### Problem 29 · faded — your turn

Return coordinate matrices built from 1-D tensors `x` and `y`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([[1., 2., 3.],
        [1., 2., 3.]]), tensor([[10., 10., 10.],
        [20., 20., 20.]]))
```


In [ ]:
import torch as t

def solve(x, y):
    """Return the tuple (X, Y) of 2-D coordinate grids."""
    return t.meshgrid(x, y, indexing=_____)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(29)


In [ ]:
#@title 💡 Solution — Problem 29
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, y):
    X, Y = t.meshgrid(x, y, indexing='xy')
    return X, Y


example_x = t.tensor([1.0, 2.0, 3.0])
example_y = t.tensor([10.0, 20.0])
print(solve(example_x, example_y))


<!-- dd:dd-q217 -->

### Problem 217 · guided

Write a function solve(block, reps_r, reps_c) that takes a 2-D PyTorch tensor block and two positive repetition counts, and returns the tensor formed by TILING the block reps_r times vertically and reps_c times horizontally — shape (block_rows * reps_r, block_cols * reps_c).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 0, 1],
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [1, 0, 1, 0]])
```


<details>
<summary>Hints</summary>

Tile a 2-D block `reps_r` times vertically and `reps_c` times horizontally.
One `block.repeat(reps_r, reps_c)` call does it — and note that this is the
*block* operation, not the per-element one.

</details>


In [ ]:
import torch as t

def solve(block, reps_r, reps_c):
    """Tile block reps_r times down and reps_c times across."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([[0, 1], [1, 0]]), 2, 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(217)


In [ ]:
#@title 💡 Solution — Problem 217
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(block, reps_r, reps_c):
    return block.tile((reps_r, reps_c))


print(solve(t.tensor([[0, 1], [1, 0]]), 2, 2))


<!-- dd:dd-q69 -->

### Problem 69 · independent

Write a function solve(x, repeats) that takes a 1-D PyTorch tensor x and an integer tensor repeats of the same length, and returns a 1-D tensor in which each element x[i] appears exactly repeats[i] times consecutively, in order. A count of 0 drops that element entirely.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 2, 2, 2, 3, 3])
```


In [ ]:
import torch as t

def solve(x, repeats):
    """Repeat each x[i] exactly repeats[i] times, in order."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example_x = t.tensor([1, 2, 3])
example_r = t.tensor([1, 3, 2])
print(solve(example_x, example_r))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(69)


In [ ]:
#@title 💡 Solution — Problem 69
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, repeats):
    return t.repeat_interleave(x, repeats)


example_x = t.tensor([1, 2, 3])
example_r = t.tensor([1, 3, 2])
print(solve(example_x, example_r))


<!-- dd:dd-q155 -->

### Problem 155 · independent

Write a function solve(values, counts) that takes a 1-D PyTorch tensor of run values and an equal-length integer tensor of run lengths, and returns the DECODED run-length-encoded sequence: values[0] repeated counts[0] times, then values[1] repeated counts[1] times, and so on. A count of 0 contributes nothing.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 1, 3, 2, 2, 2])
```


In [ ]:
import torch as t

def solve(values, counts):
    """Decode a run-length encoding into the full sequence."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([1, 3, 2]), t.tensor([2, 1, 3])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(155)


In [ ]:
#@title 💡 Solution — Problem 155
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(values, counts):
    return t.repeat_interleave(values, counts)


print(solve(t.tensor([1, 3, 2]), t.tensor([2, 1, 3])))


#### Common mistakes

- **"`x.repeat(3)` repeats each element three times."** — That is NumPy's
  `repeat`. In PyTorch `.repeat` tiles whole copies; per-element repetition is
  `t.repeat_interleave`. The two produce the same *length* from the same
  input, so length checks won't catch the mix-up.
- **"`t.meshgrid(x, y)` matches `np.meshgrid(x, y)`."** — Only with
  `indexing='xy'`. The default is `'ij'`, which gives you the transpose.


<!-- dd:dd-kp-numpy-random-generator -->

## Random numbers with a Generator

`numpy.random-generator`


Reproducible randomness in PyTorch goes through a **`Generator`** object — you
make one, then hand it to the sampling functions:

```python no-run
rng = t.Generator().manual_seed(seed)   # fixing the seed makes runs reproducible
t.rand(5, generator=rng)                # 5 uniform floats in [0, 1)
t.rand((3, 3), generator=rng)           # any shape, same convention as constructors
t.randint(0, 10, (4,), generator=rng)   # random ints in [0, 10)
t.randn((2, 2), generator=rng)          # standard-normal draws
t.randperm(n, generator=rng)            # a random shuffling of 0..n-1
```


In [ ]:
import torch as t

rng = t.Generator().manual_seed(0)
print("rand    ", t.rand(3, generator=rng))
print("randint ", t.randint(0, 10, (5,), generator=rng))
print("randn   ", t.randn(3, generator=rng))
print("randperm", t.randperm(6, generator=rng))




Note the shape of the API: unlike NumPy, where you call *methods on* the
generator (`rng.random(5)`), in PyTorch the generator is an **argument** to a
normal function (`t.rand(5, generator=rng)`). Same idea, inverted spelling.

The key mental model: a generator is a **deterministic stream**. Seeded with
the same value, it produces the same sequence forever; each draw consumes the
next chunk of the stream. That's why reproducible code (and graders, and
experiments) *passes the rng in as an argument* instead of creating one
internally: whoever owns the stream controls reproducibility. When a function
receives `rng`, use it directly — creating a new generator or reseeding inside
breaks the caller's stream.


In [ ]:
a = t.Generator().manual_seed(42)
b = t.Generator().manual_seed(42)
print("a:", t.rand(3, generator=a))
print("b:", t.rand(3, generator=b))

# a has now advanced three draws; b is replayed from the start.
print("a again:", t.rand(3, generator=a))
assert not t.equal(t.rand(3, generator=a), t.rand(3, generator=b))




The second line of that output is the whole reproducibility contract, and the
last assert is why an *extra* draw slipped in anywhere shifts every value
after it.

You will also see the **global** API in the wild (`t.manual_seed(0)`, then
plain `t.rand(shape)` with no generator) — a shared, process-level stream. It
still works and some bank drills use it, but for anything you need to
reproduce, prefer the explicit `Generator`: no hidden global state, no spooky
interference between distant modules.

A pattern this unlocks immediately: **random structure via deterministic
building blocks** — e.g. a random permutation *matrix* is just the identity
matrix with its rows shuffled: `t.eye(n)[t.randperm(n, generator=rng)]`.


In [ ]:
rng = t.Generator().manual_seed(1)
order = t.randperm(4, generator=rng)
perm = t.eye(4)[order]
print("order", order)
print(perm)

# Every row and every column still has exactly one 1 — that is what makes it
# a permutation matrix rather than just a random 0/1 grid.
assert t.equal(perm.sum(dim=0), t.ones(4))
assert t.equal(perm.sum(dim=1), t.ones(4))


Task: given a seeded generator, draw floats reproducibly; show that the same
seed replays the same stream and that consuming draws advances it.


In [ ]:
import torch as t

# Two generators, same seed -> identical streams.
rng_a = t.Generator().manual_seed(42)
rng_b = t.Generator().manual_seed(42)

first_a = t.rand(3, generator=rng_a)
first_b = t.rand(3, generator=rng_b)
assert t.equal(first_a, first_b)          # same stream position, same values
assert first_a.shape == (3,)
assert bool(((0 <= first_a) & (first_a < 1)).all())   # uniform in [0, 1)
print("rng_a first draw:", first_a)
print("rng_b first draw:", first_b)

# Each draw CONSUMES stream: the next request continues where we left off.
second_a = t.rand(3, generator=rng_a)
assert not t.equal(first_a, second_a)
print("rng_a second draw:", second_a)

# A function that receives a generator must use it as given —
# this is what "use the next n draws from rng" means in the drills.
def noisy_zeros(rng, n):
    return t.zeros(n) + t.rand(n, generator=rng)

out1 = noisy_zeros(t.Generator().manual_seed(7), 4)
out2 = noisy_zeros(t.Generator().manual_seed(7), 4)
assert t.equal(out1, out2)                # reproducible: caller owns the seed
print("seed 7, run 1:", out1)
print("seed 7, run 2:", out2)




Why each step:

1. Seeding twice with 42 and comparing draws makes "deterministic stream"
   concrete — randomness here is repeatable on demand.
2. The consumed-stream check explains grader behavior: a drill that hands you
   `rng` has pre-computed what the *next* draws will be. Reseeding or making
   your own generator inside `solve` produces different numbers and fails.
3. `noisy_zeros` is the shape of every rng-taking function you'll write:
   thread the generator through, never create one mid-function.


<!-- dd:dd-q8 -->

### Problem 8 · faded — your turn

The next n uniform floats from a generator you are handed.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.8823, 0.9150, 0.3829])
```


In [ ]:
import torch as t

def solve(rng, n):
    """Return the next n uniform [0,1) floats from rng's stream."""
    return t.rand(n, _____=rng)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(8)


In [ ]:
#@title 💡 Solution — Problem 8
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rng, n):
    return t.rand(n, generator=rng)


example_rng = t.Generator().manual_seed(42)
print(solve(example_rng, 3))


<!-- dd:dd-q42 -->

### Problem 42 · guided

Write a function solve(shape) that takes a tuple of three integers, e.g. (3, 3, 3), and returns a new PyTorch tensor of exactly that shape filled with uniform random floats drawn from the interval [0, 1). Each element must be an independent draw, so a multi-element result should not be constant. The grader checks the returned tensor's shape, floating dtype, value range, and that the values actually vary.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0.4963, 0.7682, 0.0885],
         [0.1320, 0.3074, 0.6341],
         [0.4901, 0.8964, 0.4556]],

        [[0.6323, 0.3489, 0.4017],
         [0.0223, 0.1689, 0.2939],
         [0.5185, 0.6977, 0.8000]],

        [[0.1610, 0.2823, 0.6816],
         [0.9152, 0.3971, 0.8742],
         [0.4194, 0.5529, 0.9527]]])
```


<details>
<summary>Hints</summary>

1. A random tensor of a given 3-D shape, uniform in [0, 1) — this drill uses
   the global stream, so no generator argument is needed.
2. The sampling functions share the shape convention with constructors: pass
   the whole tuple.
3. `t.rand(shape)` — the grader checks shape, dtype, range, and that values
   vary (so no constant tensors).

</details>


In [ ]:
import torch as t

def solve(shape):
    """Return a tensor of the given shape filled with uniform floats in [0, 1)."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
t.manual_seed(0)
example = (3, 3, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(42)


In [ ]:
#@title 💡 Solution — Problem 42
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(shape):
    return t.rand(shape)


t.manual_seed(0)
example = (3, 3, 3)
print(solve(example))


<!-- dd:dd-q104 -->

### Problem 104 · independent

Write a function solve(n, rng) that takes a size n and a torch.Generator rng, and returns an n x n floating-point permutation matrix: every row and every column contains exactly one 1.0 with all other entries 0.0, the arrangement chosen at random using rng. The grader checks the permutation-matrix properties, so any valid random permutation passes.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0., 0., 1.],
        [1., 0., 0.],
        [0., 1., 0.]])
```


In [ ]:
import torch as t

def solve(n, rng):
    """Return a random n x n permutation matrix drawn using rng."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(3, t.Generator().manual_seed(0)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(104)


In [ ]:
#@title 💡 Solution — Problem 104
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n, rng):
    return t.eye(n)[t.randperm(n, generator=rng)]


print(solve(3, t.Generator().manual_seed(0)))


#### Common mistakes

- **"I should reseed inside my function for safety."** — The opposite: a
  function that receives `rng` must use the caller's stream. Reseeding or
  creating a new generator inside breaks reproducibility (and fails graders
  that check the exact stream).
- **"`rng.rand(3)` — the generator has the sampling methods."** — It does not.
  The generator is passed *to* `t.rand(...)` as `generator=`. Its own methods
  are about seed and state.
- **"`t.rand(3)` might include 1.0."** — The interval is half-open
  [0, 1): 0 is possible, 1 is not. Scale/shift for other ranges.
- **"Same code, same randomness."** — Only with the same SEED and the same
  DRAW ORDER. Any extra draw in between shifts everything after it; that's
  what "each draw consumes the stream" means.


<!-- dd:dd-kp-numpy-linalg-basics -->

## Matrix multiply and t.linalg basics

`numpy.linalg-basics`


### two multiplications — * vs @


Two different "multiplications" exist for matrices, and PyTorch gives each its
own operator:

- **`a * b` — elementwise**: multiplies corresponding entries; shapes must
  match (or broadcast). No summing happens.
- **`a @ b` — matrix multiplication**: row-times-column with a sum inside.
  For `a` of shape (m, k) and `b` of shape (k, n), the result is (m, n):
  entry `[i, j]` is the dot product of row i of `a` with column j of `b`.
  The inner dimensions (k) must agree, and they disappear in the output.


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print("a * b (elementwise)")
print(a * b)
print("a @ b (matrix product)")
print(a @ b)




`a*b` entry [0,0] is 1·5. `a@b` entry [0,0] is 1·5 + 2·7 = 19 — the row met
the column and the k axis was summed away.

The shape rule `(m, k) @ (k, n) → (m, n)` is worth chanting: it predicts
both whether a product is legal and what comes out. It also covers
matrix–vector: `(m, k) @ (k,) → (m,)`.


In [ ]:
print((t.ones((2, 3)) @ t.ones((3, 4))).shape)     # (2,4): the 3s vanish
print((t.ones((2, 3)) @ t.ones(3)).shape)          # (2,): matrix times vector

try:
    t.ones((2, 3)) @ t.ones((2, 3))                # inner dims 3 vs 2
except RuntimeError as err:
    print("RuntimeError:", err)




`t.matmul(a, b)` is the same operation spelled as a function, and in model
code you will meet `a.T` for the transpose that so often precedes it.


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
b = t.tensor([[5.0, 6.0],
              [7.0, 8.0]])

# Elementwise vs matrix product — same operands, different operations:
elem = a * b            # [[5, 12], [21, 32]] — corresponding entries
mat = a @ b             # row·column with a sum inside
assert elem.tolist() == [[5.0, 12.0], [21.0, 32.0]]
assert mat.tolist() == [[19.0, 22.0], [43.0, 50.0]]
# Check one entry by hand: mat[0,0] = 1*5 + 2*7 = 19. Row 0 · column 0.
print("a * b")
print(elem)
print("a @ b")
print(mat)

# Shape rule: (2,3) @ (3,2) -> (2,2); the inner 3s must match and vanish.
p = t.ones((2, 3)) @ t.ones((3, 2))
assert p.shape == (2, 2)




Why: computing `mat[0, 0]` by hand once (row 0 of `a` dotted with column 0
of `b`) is the fastest way to internalize what `@` does beyond the shape
rule — and predicting shapes BEFORE running makes mismatches design errors
you catch on paper.


<!-- dd:dd-q239 -->

### Problem 239 · faded — your turn

Matrix product of shapes (m, k) and (k, n).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[19., 22.],
        [43., 50.]])
```


In [ ]:
import torch as t

def solve(a, b):
    """The (m, n) matrix product of a (m, k) and b (k, n)."""
    return a _____ b


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(239)


In [ ]:
#@title 💡 Solution — Problem 239
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return a @ b


a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(a, b))


### t.linalg.solve — never build the inverse


The `t.linalg` submodule holds the "real linear algebra", and its names match
NumPy's `np.linalg` almost one for one:

- **`t.linalg.solve(a, b)`** — solve the system `a @ x = b` for `x`.
  This is THE way to compute "a⁻¹ b". Numerically, solving directly is both
  faster and more accurate than `t.linalg.inv(a) @ b`; computing an
  explicit inverse is almost never what you want.
- `t.linalg.inv`, `t.linalg.det`, `t.linalg.matrix_rank`,
  `t.linalg.norm`, `t.linalg.eig` — inverse, determinant, rank, norms,
  eigendecomposition, when a task genuinely asks for them.

One dtype caveat that is easy to trip over here: these routines want floats,
and the default float is 32-bit. Ill-conditioned systems lose accuracy sooner
than the float64 you may be used to from NumPy — if a solve looks wrong,
checking the dtype is a reasonable first move.


In [ ]:
import torch as t

a_sys = t.tensor([[3.0, 1.0],
                  [1.0, 2.0]])
b_vec = t.tensor([9.0, 8.0])
x = t.linalg.solve(a_sys, b_vec)
print("x =", x)




Sanity-checking a solve is one line: plug `x` back in and compare
`a @ x` with `b` using `t.allclose` (float arithmetic — never `==`).


In [ ]:
print("a @ x =", a_sys @ x, " b =", b_vec)
print("exactly equal? ", bool(t.equal(a_sys @ x, b_vec)))
print("close enough?  ", bool(t.allclose(a_sys @ x, b_vec)))
assert t.allclose(a_sys @ x, b_vec)




The inverse route reaches the same answer and does more work to get there —
run it once so the equivalence is concrete, then stop writing it:


In [ ]:
via_inverse = t.linalg.inv(a_sys) @ b_vec
print("solve:  ", x)
print("inverse:", via_inverse)
assert t.allclose(x, via_inverse)


In [ ]:
import torch as t

# Solve a @ x = b_vec — NOT by computing an inverse.
a_sys = t.tensor([[2.0, 0.0],
                  [0.0, 4.0]])
b_vec = t.tensor([6.0, 8.0])
x = t.linalg.solve(a_sys, b_vec)
assert x.tolist() == [3.0, 2.0]
print("x =", x)

# Verification pattern: substitute back, compare with float tolerance.
assert t.allclose(a_sys @ x, b_vec)
print("a @ x =", a_sys @ x, " b =", b_vec)




Why: `solve` + `allclose` verification — the pair costs one line and
catches both wrong answers and ill-conditioned systems.


<!-- dd:dd-q107 -->

### Problem 107 · faded — your turn

Solve the linear system a @ x = b (a is invertible).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([3., 2.])
```


In [ ]:
import torch as t

def solve(a, b):
    """Return x such that a @ x = b (use a solver, not an inverse)."""
    return t.linalg._____(a, b)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(107)


In [ ]:
#@title 💡 Solution — Problem 107
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.linalg.solve(a, b)


example_a = t.tensor([[2.0, 0.0], [0.0, 4.0]])
example_b = t.tensor([6.0, 8.0])
print(solve(example_a, example_b))


<!-- dd:dd-q508 -->

### Problem 508 · guided

Write a function solve(a, b) that takes two float matrices of the same shape and returns their elementwise product — entry [i][j] is a[i][j] * b[i][j]. This is the multiplication that is NOT matrix multiplication, and telling the two apart is the whole point.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 5., 12.],
        [21., 32.]])
```


<details>
<summary>Hints</summary>

1. This is the multiplication that is NOT matrix multiplication.
2. Entry [i][j] depends only on the two entries at [i][j] — nothing is summed,
   so no axis is contracted.
3. `a * b`.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return the ELEMENTWISE product of two same-shaped matrices."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([[5.0, 6.0], [7.0, 8.0]]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(508)


In [ ]:
#@title 💡 Solution — Problem 508
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Return the ELEMENTWISE product of two same-shaped matrices."""
    return a * b


example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([[5.0, 6.0], [7.0, 8.0]]))
print(solve(*example))


<!-- dd:dd-q509 -->

### Problem 509 · guided

Write a function solve(a, b) that takes two square float matrices and returns a tuple (elementwise, matmul) of plain nested lists: first a * b, then a @ b. Run them side by side once and the difference stops being something you have to remember.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[5.0, 12.0], [21.0, 32.0]], [[19.0, 22.0], [43.0, 50.0]])
```


<details>
<summary>Hints</summary>

1. Both answers come from the same two matrices; only the operator changes.
2. `*` pairs entries in place; `@` contracts a's columns against b's rows.
3. `((a * b).tolist(), (a @ b).tolist())`.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return (elementwise product, matrix product) for two square matrices."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([[5.0, 6.0], [7.0, 8.0]]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(509)


In [ ]:
#@title 💡 Solution — Problem 509
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Return (elementwise product, matrix product) for two square matrices."""
    return ((a * b).tolist(), (a @ b).tolist())


example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([[5.0, 6.0], [7.0, 8.0]]))
print(solve(*example))


<!-- dd:dd-q510 -->

### Problem 510 · independent

Write a function solve(a) that returns a tuple (values, shape) for the transpose of the 2-D tensor a. A transpose only re-describes the block — the numbers never move — so the shape comes back with its two axes swapped.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[1.0, 4.0], [2.0, 5.0], [3.0, 6.0]], (3, 2))
```


In [ ]:
import torch as t

def solve(a):
    """Return the transpose of a 2-D tensor, and its shape."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(510)


In [ ]:
#@title 💡 Solution — Problem 510
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a):
    """Return the transpose of a 2-D tensor, and its shape."""
    tr = a.T
    return (tr.tolist(), tuple(tr.shape))


example = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(solve(example))


<!-- dd:dd-q511 -->

### Problem 511 · independent

Write a function solve(a, x) where a has shape (m, n) and x has shape (n,), returning a tuple (values, shape) for the matrix-vector product. The shared axis vanishes, so an (m, n) against an (n,) leaves you with (m,) — one number per row.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([3.0, 7.0], (2,))
```


In [ ]:
import torch as t

def solve(a, x):
    """Return a @ x for a matrix and a vector, plus the result shape."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([1.0, 1.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(511)


In [ ]:
#@title 💡 Solution — Problem 511
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, x):
    """Return a @ x for a matrix and a vector, plus the result shape."""
    r = a @ x
    return (r.tolist(), tuple(r.shape))


example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([1.0, 1.0]))
print(solve(*example))


<!-- dd:dd-q512 -->

### Problem 512 · independent

Write a function solve(a, b) that solves the linear system a @ x = b for x, and returns a tuple (x_rounded, checks_out): x's entries rounded to 4 decimal places as a plain list, and a plain bool saying whether a @ x really does reproduce b. Solve the system directly — never build the inverse and multiply, which is slower and less accurate for the same answer.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([3.0, 2.0], True)
```


In [ ]:
import torch as t

def solve(a, b):
    """Solve a @ x = b, and confirm the solution satisfies it."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[2.0, 0.0], [0.0, 4.0]]), t.tensor([6.0, 8.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(512)


In [ ]:
#@title 💡 Solution — Problem 512
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Solve a @ x = b, and confirm the solution satisfies it."""
    x = t.linalg.solve(a, b)
    return ([round(v, 4) for v in x.tolist()], bool(t.allclose(a @ x, b, atol=1e-4)))


example = (t.tensor([[2.0, 0.0], [0.0, 4.0]]), t.tensor([6.0, 8.0]))
print(solve(*example))


<!-- dd:dd-q513 -->

### Problem 513 · independent

Write a function solve(mats, x) where mats has shape (batch, m, n) and x has shape (n,), returning a tuple (values, shape) for the result of applying every matrix in the batch to x. @ treats all but the last two axes as batch dimensions, so this is one operator, not a loop.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[3.0, 4.0], [6.0, 8.0]], (2, 2))
```


In [ ]:
import torch as t

def solve(mats, x):
    """Apply a BATCH of matrices to one vector."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[[1.0, 0.0], [0.0, 1.0]], [[2.0, 0.0], [0.0, 2.0]]]), t.tensor([3.0, 4.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(513)


In [ ]:
#@title 💡 Solution — Problem 513
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(mats, x):
    """Apply a BATCH of matrices to one vector."""
    r = mats @ x
    return (r.tolist(), tuple(r.shape))


example = (t.tensor([[[1.0, 0.0], [0.0, 1.0]], [[2.0, 0.0], [0.0, 2.0]]]), t.tensor([3.0, 4.0]))
print(solve(*example))


#### Common mistakes

- **"`*` multiplies matrices."** — `*` is elementwise; `@` is the matrix
  product. Mixing them up usually *doesn't* crash (broadcasting can make `*`
  legal), it just silently computes the wrong thing — the worst kind of bug.
- **"To solve a @ x = b, compute inv(a) @ b."** — `t.linalg.solve(a, b)` is
  more accurate and faster; explicit inverses amplify rounding error and cost
  more. Reach for `inv` only when the inverse itself is the deliverable.
- **"If `@` runs, the shapes were right."** — `@` between wrong-but-compatible
  shapes (e.g. transposed operands, square matrices) runs happily and returns
  garbage. Predict `(m, k) @ (k, n) → (m, n)` on paper first.
- **"Integer tensors are fine for linalg."** — They are not; the solvers
  require floating point and will raise. Convert with `.to(t.float32)` first.
